<a href="https://colab.research.google.com/github/mena-04/qml/blob/main/phases2_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports and setup

In [1]:

%pip install -q pennylane scikit-learn pandas matplotlib


In [2]:

import numpy as np
import pandas as pd
import pennylane as qml
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import rbf_kernel

SEED = 42
N_QUBITS = 4

np.random.seed(SEED)

print("PennyLane:", qml.__version__)


PennyLane: 0.45.1


Load the existing training subset

In [3]:

from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()

zip_path = next(
    name for name in uploaded
    if name.endswith(".zip")
)

with zipfile.ZipFile(zip_path) as z:
    z.extractall(".")

data_path = Path(
    "limited_data/data_n80_seed42.npz"
)

with np.load(data_path) as d:
    X_train = d["X_train"].copy()
    y_train = d["y_train"].copy()
    train_idx = d["train_idx"].copy()

print("Training shape:", X_train.shape)
print("Malignant samples:", sum(y_train == 1))
print("Benign samples:", sum(y_train == 0))


Saving qbreast_limited_data.zip to qbreast_limited_data.zip
Training shape: (80, 4)
Malignant samples: 30
Benign samples: 50


Define the quantum feature maps

In [4]:

dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev)
def simple_ry_state(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)
    return qml.state()


@qml.qnode(dev)
def reupload_state(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    for i in range(N_QUBITS):
        qml.RZ(x[i], wires=i)
        qml.RY(x[i], wires=i)

    return qml.state()


def state_kernel(X, state_fn):
    states = np.asarray(
        [state_fn(x) for x in X],
        dtype=np.complex128
    )

    return np.abs(
        states @ states.conj().T
    ) ** 2


Calculate the kernels

In [5]:

K_classical = rbf_kernel(
    X_train,
    gamma=0.1
)

K_simple = state_kernel(
    X_train,
    simple_ry_state
)

K_reupload = state_kernel(
    X_train,
    reupload_state
)

kernels = {
    "Classical RBF": K_classical,
    "Simple RY": K_simple,
    "Data Reuploading": K_reupload
}

for name, K in kernels.items():
    print(
        name,
        "| Shape:", K.shape,
        "| Symmetric:", np.allclose(K, K.T),
        "| Min eigenvalue:",
        np.linalg.eigvalsh(K).min()
    )


Classical RBF | Shape: (80, 80) | Symmetric: True | Min eigenvalue: 1.2244440468732379e-09
Simple RY | Shape: (80, 80) | Symmetric: True | Min eigenvalue: 1.3550071450654595e-13
Data Reuploading | Shape: (80, 80) | Symmetric: True | Min eigenvalue: 1.1287028124441463e-06


Kernel-target alignment

In [6]:

def center_kernel(K):
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    return H @ K @ H


def kernel_target_alignment(K, y):
    # Encode labels as -1 and +1
    labels = 2 * y - 1

    target = np.outer(labels, labels)

    Kc = center_kernel(K)
    Tc = center_kernel(target)

    numerator = np.sum(Kc * Tc)

    denominator = (
        np.linalg.norm(Kc, "fro")
        * np.linalg.norm(Tc, "fro")
    )

    return numerator / denominator


alignment_results = []

for name, K in kernels.items():
    alignment_results.append({
        "kernel": name,
        "alignment": kernel_target_alignment(
            K, y_train
        )
    })

alignment_df = pd.DataFrame(
    alignment_results
)

display(alignment_df.round(4))


,kernel,alignment
0,Classical RBF,0.3698
1,Simple RY,0.3745
2,Data Reuploading,0.4181


Effective rank

In [7]:

def effective_rank(K):
    eigenvalues = np.linalg.eigvalsh(K)

    eigenvalues = np.clip(
        eigenvalues, 0, None
    )

    probabilities = (
        eigenvalues / eigenvalues.sum()
    )

    probabilities = probabilities[
        probabilities > 1e-12
    ]

    entropy = -np.sum(
        probabilities * np.log(probabilities)
    )

    return np.exp(entropy)


rank_results = []

for name, K in kernels.items():
    rank_results.append({
        "kernel": name,
        "effective_rank": effective_rank(K)
    })

rank_df = pd.DataFrame(rank_results)

display(rank_df.round(4))


,kernel,effective_rank
0,Classical RBF,2.2294
1,Simple RY,4.1274
2,Data Reuploading,7.9801


Compare kernel similarity

In [8]:

def kernel_similarity(K1, K2):
    A = center_kernel(K1)
    B = center_kernel(K2)

    return np.sum(A * B) / (
        np.linalg.norm(A, "fro")
        * np.linalg.norm(B, "fro")
    )


similarity_matrix = pd.DataFrame(
    index=kernels.keys(),
    columns=kernels.keys(),
    dtype=float
)

for name1, K1 in kernels.items():
    for name2, K2 in kernels.items():
        similarity_matrix.loc[
            name1, name2
        ] = kernel_similarity(K1, K2)

display(similarity_matrix.round(4))


,Classical RBF,Simple RY,Data Reuploading
Classical RBF,1.0000,0.9725,0.6837
Simple RY,0.9725,1.0000,0.7584
Data Reuploading,0.6837,0.7584,1.0000


Save Phase 2 diagnostics

In [9]:

from pathlib import Path

output = Path("phase_2_results")
output.mkdir(exist_ok=True)

alignment_df.to_csv(
    output / "kernel_alignment.csv",
    index=False
)

rank_df.to_csv(
    output / "effective_rank.csv",
    index=False
)

similarity_matrix.to_csv(
    output / "kernel_similarity.csv"
)

print("Phase 2 diagnostic results saved.")


Phase 2 diagnostic results saved.


Trainable Quantum Feature Map

In [10]:

import numpy as np
import pennylane as qml

from scipy.optimize import minimize

N_QUBITS = 4

dev_trainable = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(dev_trainable)
def trainable_state(x, theta):

    # First data encoding
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

    # Entangling layer
    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    # Trainable data re-uploading
    for i in range(N_QUBITS):
        qml.RY(theta[i] * x[i], wires=i)

    return qml.state()


def trainable_kernel(Xa, Xb, theta):

    states_a = np.asarray([
        trainable_state(x, theta)
        for x in Xa
    ], dtype=np.complex128)

    states_b = np.asarray([
        trainable_state(x, theta)
        for x in Xb
    ], dtype=np.complex128)

    return np.abs(
        states_a @ states_b.conj().T
    ) ** 2


Trainability Sanity Check

In [11]:

X_small = X_train[:12]

theta_a = np.ones(4)
theta_b = np.array([0.5, 1.5, 0.8, 1.2])

K_a = trainable_kernel(
    X_small, X_small, theta_a
)

K_b = trainable_kernel(
    X_small, X_small, theta_b
)

difference = np.max(
    np.abs(K_a - K_b)
)

print("Maximum kernel difference:", difference)

assert difference > 1e-6

print("Trainable parameters affect the kernel.")


Maximum kernel difference: 0.2207431649900715
Trainable parameters affect the kernel.


Kernel-Target Alignment Objective

In [12]:

def alignment_objective(theta, X, y):

    K = trainable_kernel(
        X, X, theta
    )

    alignment = kernel_target_alignment(
        K, y
    )

    return -float(alignment)


Optimize Quantum Kernel Alignment

In [13]:

import time

# First optimization on 40 of the 80 training samples
rng = np.random.default_rng(42)

subset_indices = rng.choice(
    len(X_train),
    size=40,
    replace=False
)

X_opt = X_train[subset_indices]
y_opt = y_train[subset_indices]

theta_initial = np.ones(4)

initial_alignment = -alignment_objective(
    theta_initial, X_opt, y_opt
)

print("Initial alignment:", initial_alignment)

start = time.perf_counter()

optimization = minimize(
    alignment_objective,
    x0=theta_initial,
    args=(X_opt, y_opt),
    method="L-BFGS-B",
    bounds=[(0.1, 2.0)] * 4,
    options={
        "maxiter": 30,
        "ftol": 1e-6
    }
)

elapsed = time.perf_counter() - start

theta_optimized = optimization.x

final_alignment = -alignment_objective(
    theta_optimized, X_opt, y_opt
)

print("Optimization success:", optimization.success)
print("Optimization message:", optimization.message)
print("Optimized parameters:", theta_optimized)
print("Initial alignment:", initial_alignment)
print("Final alignment:", final_alignment)
print("Optimization time:", elapsed)


Initial alignment: 0.30277129452500684
Optimization success: True
Optimization message: CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
Optimized parameters: [0.96182358 0.1        0.1        0.1       ]
Initial alignment: 0.30277129452500684
Final alignment: 0.4170461680038129
Optimization time: 14.554338820999874


Validate the Learned Kernel

In [14]:

K_optimized = trainable_kernel(
    X_opt, X_opt, theta_optimized
)

eigenvalues = np.linalg.eigvalsh(
    K_optimized
)

print(
    "Symmetric:",
    np.allclose(K_optimized, K_optimized.T)
)

print(
    "Diagonal near 1:",
    np.allclose(np.diag(K_optimized), 1)
)

print(
    "Minimum eigenvalue:",
    eigenvalues.min()
)

assert np.allclose(
    K_optimized, K_optimized.T,
    atol=1e-8
)

assert np.allclose(
    np.diag(K_optimized), 1,
    atol=1e-8
)

assert eigenvalues.min() > -1e-8

print("Learned quantum kernel passed validation.")


Symmetric: True
Diagonal near 1: True
Minimum eigenvalue: 1.3385135962725305e-06
Learned quantum kernel passed validation.


Alignment Improvement

In [16]:

alignment_comparison = pd.DataFrame([
    {
        "model": "Untrained Quantum Kernel",
        "alignment": initial_alignment
    },
    {
        "model": "Trainable Quantum Kernel",
        "alignment": final_alignment
    }
])

alignment_comparison["improvement"] = (
    alignment_comparison["alignment"]
    - initial_alignment
)

display(alignment_comparison.round(5))


,model,alignment,improvement
0,Untrained Quantum Kernel,0.30277,0.00000
1,Trainable Quantum Kernel,0.41705,0.11427


Leakage-Safe Trainable Kernel Evaluation

In [17]:

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score
)
from sklearn.metrics.pairwise import rbf_kernel
from scipy.optimize import minimize

cv_phase2 = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

MODEL_NAMES = [
    "Classical RBF",
    "Simple RY",
    "Fixed Reuploading",
    "Trainable Quantum"
]

validation_records = []


Compare Fixed and Trainable Kernels

In [18]:

import time

# Load raw data and indices from Phase 2.1
dataset = load_breast_cancer()
X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

X_subset_raw = X_raw[train_idx]
y_subset = y_all[train_idx]

validation_records = []

for fold, (tr, val) in enumerate(
    cv_phase2.split(X_subset_raw, y_subset)
):

    prep = Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=4)),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        ))
    ])

    Xtr = prep.fit_transform(X_subset_raw[tr])
    Xval = prep.transform(X_subset_raw[val])

    ytr = y_subset[tr]
    yval = y_subset[val]

    # Train trainable kernel on fold-training data only
    opt = minimize(
        alignment_objective,
        x0=np.ones(4),
        args=(Xtr, ytr),
        method="L-BFGS-B",
        bounds=[(0.1, 2.0)] * 4,
        options={"maxiter": 30}
    )

    theta = opt.x

    for model_name in MODEL_NAMES:

        start = time.perf_counter()

        if model_name == "Classical RBF":
            Ktr = rbf_kernel(Xtr, gamma=0.1)
            Kval = rbf_kernel(
                Xval, Xtr, gamma=0.1
            )

        elif model_name == "Simple RY":
            Ktr = state_kernel(
                Xtr, simple_ry_state
            )

            states_val = np.asarray([
                simple_ry_state(x) for x in Xval
            ])
            states_tr = np.asarray([
                simple_ry_state(x) for x in Xtr
            ])

            Kval = np.abs(
                states_val @ states_tr.conj().T
            ) ** 2

        elif model_name == "Fixed Reuploading":
            Ktr = state_kernel(
                Xtr, reupload_state
            )

            states_val = np.asarray([
                reupload_state(x) for x in Xval
            ])
            states_tr = np.asarray([
                reupload_state(x) for x in Xtr
            ])

            Kval = np.abs(
                states_val @ states_tr.conj().T
            ) ** 2

        else:
            Ktr = trainable_kernel(
                Xtr, Xtr, theta
            )

            Kval = trainable_kernel(
                Xval, Xtr, theta
            )

        model = SVC(
            kernel="precomputed",
            C=1.0
        )

        model.fit(Ktr, ytr)

        pred = model.predict(Kval)
        scores = model.decision_function(Kval)

        validation_records.append({
            "fold": fold,
            "model": model_name,
            "roc_auc": roc_auc_score(
                yval, scores
            ),
            "recall": recall_score(
                yval, pred
            ),
            "f1": f1_score(
                yval, pred,
                zero_division=0
            ),
            "runtime_seconds": (
                time.perf_counter() - start
            )
        })

validation_df = pd.DataFrame(
    validation_records
)

display(validation_df.round(4))


,fold,model,roc_auc,recall,f1,runtime_seconds
0,0,Classical RBF,0.9882,0.5,0.6667,0.0256
1,0,Simple RY,0.9765,0.5,0.6667,0.1555
2,0,Fixed Reuploading,0.8647,0.6,0.7500,0.3790
3,0,Trainable Quantum,0.9941,0.6,0.7500,0.4063
4,1,Classical RBF,0.9765,0.6,0.7500,0.0077
5,1,Simple RY,0.9588,0.7,0.7778,0.1687
6,1,Fixed Reuploading,0.9824,0.7,0.8235,0.3567
7,1,Trainable Quantum,0.9588,0.7,0.7778,0.4100
8,2,Classical RBF,1.0000,1.0,1.0000,0.0093
9,2,Simple RY,1.0000,1.0,0.9524,0.3722


Validation Performance Summary

In [19]:

phase2_summary = (
    validation_df
    .groupby("model")
    .agg(
        auc_mean=("roc_auc", "mean"),
        auc_std=("roc_auc", "std"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean")
    )
    .reset_index()
)

display(
    phase2_summary.sort_values(
        "auc_mean",
        ascending=False
    ).round(4)
)

validation_df.to_csv(
    "phase2_trainable_validation.csv",
    index=False
)

phase2_summary.to_csv(
    "phase2_trainable_summary.csv",
    index=False
)


,model,auc_mean,auc_std,recall_mean,f1_mean
0,Classical RBF,0.9882,0.0118,0.7000,0.8056
3,Trainable Quantum,0.9843,0.0223,0.7667,0.8267
2,Simple RY,0.9784,0.0207,0.7333,0.7989
1,Fixed Reuploading,0.9407,0.0659,0.7667,0.8420


Repeated Cross-Validation Setup

In [20]:

import numpy as np
import pandas as pd
import time

from pathlib import Path
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score
)
from sklearn.metrics.pairwise import rbf_kernel

TRAIN_SIZES = [40, 80, 160]
SEEDS = [42, 123, 2026]

MODEL_NAMES = [
    "Classical RBF",
    "Simple RY",
    "Fixed Reuploading",
    "Trainable Quantum"
]

C_FIXED = 1.0
MAX_ITER = 30

phase24_records = []
optimization_records = []

print("Phase 2.4 configuration ready.")


Phase 2.4 configuration ready.


Kernel Computation Helpers

In [21]:

def compute_state_matrix(X, state_fn):
    return np.asarray(
        [state_fn(x) for x in X],
        dtype=np.complex128
    )


def quantum_kernel_pair(Xtr, Xval, state_fn):
    states_tr = compute_state_matrix(
        Xtr, state_fn
    )

    states_val = compute_state_matrix(
        Xval, state_fn
    )

    Ktr = np.abs(
        states_tr @ states_tr.conj().T
    ) ** 2

    Kval = np.abs(
        states_val @ states_tr.conj().T
    ) ** 2

    return Ktr, Kval


def trainable_kernel_pair(Xtr, Xval, theta):
    states_tr = np.asarray(
        [trainable_state(x, theta) for x in Xtr],
        dtype=np.complex128
    )

    states_val = np.asarray(
        [trainable_state(x, theta) for x in Xval],
        dtype=np.complex128
    )

    Ktr = np.abs(
        states_tr @ states_tr.conj().T
    ) ** 2

    Kval = np.abs(
        states_val @ states_tr.conj().T
    ) ** 2

    return Ktr, Kval


Multi-Seed Kernel Benchmark

In [22]:

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler
)
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

dataset = load_breast_cancer()
X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

def phase24_preprocessor():
    return Pipeline([
        ("standardize", StandardScaler()),
        ("pca", PCA(n_components=4)),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        ))
    ])

phase24_records = []
optimization_records = []

for n_samples in TRAIN_SIZES:
    for seed in SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            subset_idx = d["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold_id, (tr, val) in enumerate(
            cv.split(X_subset, y_subset)
        ):

            prep = phase24_preprocessor()

            Xtr = prep.fit_transform(
                X_subset[tr]
            )
            Xval = prep.transform(
                X_subset[val]
            )

            ytr = y_subset[tr]
            yval = y_subset[val]

            # Optimize quantum kernel using fold-training only
            theta0 = np.ones(4)

            initial_alignment = (
                -alignment_objective(
                    theta0, Xtr, ytr
                )
            )

            opt_start = time.perf_counter()

            opt = minimize(
                alignment_objective,
                x0=theta0,
                args=(Xtr, ytr),
                method="L-BFGS-B",
                bounds=[(0.1, 2.0)] * 4,
                options={"maxiter": MAX_ITER}
            )

            opt_seconds = (
                time.perf_counter() - opt_start
            )

            theta = opt.x

            final_alignment = (
                -alignment_objective(
                    theta, Xtr, ytr
                )
            )

            optimization_records.append({
                "n_samples": n_samples,
                "seed": seed,
                "fold": fold_id,
                "initial_alignment": initial_alignment,
                "final_alignment": final_alignment,
                "alignment_gain": (
                    final_alignment - initial_alignment
                ),
                "optimization_seconds": opt_seconds,
                "optimization_success": opt.success,
                "theta": theta.tolist(),
                "parameters_at_lower_bound": int(
                    np.sum(
                        np.isclose(theta, 0.1, atol=1e-4)
                    )
                )
            })

            for model_name in MODEL_NAMES:

                start = time.perf_counter()

                if model_name == "Classical RBF":
                    Ktr = rbf_kernel(
                        Xtr, gamma=0.1
                    )
                    Kval = rbf_kernel(
                        Xval, Xtr, gamma=0.1
                    )

                elif model_name == "Simple RY":
                    Ktr, Kval = quantum_kernel_pair(
                        Xtr, Xval, simple_ry_state
                    )

                elif model_name == "Fixed Reuploading":
                    Ktr, Kval = quantum_kernel_pair(
                        Xtr, Xval, reupload_state
                    )

                else:
                    Ktr, Kval = trainable_kernel_pair(
                        Xtr, Xval, theta
                    )

                model = SVC(
                    kernel="precomputed",
                    C=C_FIXED
                )

                model.fit(Ktr, ytr)

                predictions = model.predict(Kval)
                scores = model.decision_function(Kval)

                phase24_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold_id,
                    "model": model_name,
                    "roc_auc": roc_auc_score(
                        yval, scores
                    ),
                    "recall": recall_score(
                        yval, predictions,
                        zero_division=0
                    ),
                    "f1": f1_score(
                        yval, predictions,
                        zero_division=0
                    ),
                    "evaluation_seconds": (
                        time.perf_counter() - start
                    ),
                    "optimization_seconds": (
                        opt_seconds
                        if model_name == "Trainable Quantum"
                        else 0.0
                    )
                })

            print(
                f"Completed n={n_samples}, "
                f"seed={seed}, fold={fold_id}"
            )

        # Save after every seed
        pd.DataFrame(phase24_records).to_csv(
            "phase24_validation.csv",
            index=False
        )

        pd.DataFrame(optimization_records).to_csv(
            "phase24_optimization.csv",
            index=False
        )

phase24_df = pd.DataFrame(phase24_records)
optimization_df = pd.DataFrame(optimization_records)

print("All repeated validation experiments completed.")


Completed n=40, seed=42, fold=0
Completed n=40, seed=42, fold=1
Completed n=40, seed=42, fold=2
Completed n=40, seed=123, fold=0
Completed n=40, seed=123, fold=1
Completed n=40, seed=123, fold=2
Completed n=40, seed=2026, fold=0
Completed n=40, seed=2026, fold=1
Completed n=40, seed=2026, fold=2
Completed n=80, seed=42, fold=0
Completed n=80, seed=42, fold=1
Completed n=80, seed=42, fold=2
Completed n=80, seed=123, fold=0
Completed n=80, seed=123, fold=1
Completed n=80, seed=123, fold=2
Completed n=80, seed=2026, fold=0
Completed n=80, seed=2026, fold=1
Completed n=80, seed=2026, fold=2
Completed n=160, seed=42, fold=0
Completed n=160, seed=42, fold=1
Completed n=160, seed=42, fold=2
Completed n=160, seed=123, fold=0
Completed n=160, seed=123, fold=1
Completed n=160, seed=123, fold=2
Completed n=160, seed=2026, fold=0
Completed n=160, seed=2026, fold=1
Completed n=160, seed=2026, fold=2
All repeated validation experiments completed.


Aggregate Validation Performance

In [23]:

# Average the three folds within each seed
per_seed = (
    phase24_df
    .groupby(["n_samples", "seed", "model"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        evaluation_seconds=("evaluation_seconds", "sum"),
        optimization_seconds=("optimization_seconds", "sum")
    )
    .reset_index()
)

# Then summarize across the three seeds
phase24_summary = (
    per_seed
    .groupby(["n_samples", "model"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        evaluation_seconds_mean=(
            "evaluation_seconds", "mean"
        ),
        optimization_seconds_mean=(
            "optimization_seconds", "mean"
        )
    )
    .reset_index()
)

display(phase24_summary.round(4))

per_seed.to_csv(
    "phase24_per_seed.csv",
    index=False
)

phase24_summary.to_csv(
    "phase24_summary.csv",
    index=False
)


,n_samples,model,auc_mean,auc_std,recall_mean,recall_std,f1_mean,f1_std,evaluation_seconds_mean,optimization_seconds_mean
0,40,Classical RBF,0.9633,0.0282,0.6667,0.1333,0.7438,0.0990,0.0347,0.0000
1,40,Fixed Reuploading,0.9552,0.0104,0.7778,0.0385,0.8142,0.0121,0.4199,0.0000
2,40,Simple RY,0.9469,0.0333,0.8000,0.0667,0.8305,0.0453,0.1975,0.0000
3,40,Trainable Quantum,0.9410,0.0474,0.7556,0.1388,0.8048,0.1065,0.3550,30.9832
4,80,Classical RBF,0.9878,0.0022,0.7556,0.0509,0.8531,0.0423,0.0271,0.0000
5,80,Fixed Reuploading,0.9675,0.0242,0.8000,0.0333,0.8444,0.0175,0.6723,0.0000
6,80,Simple RY,0.9880,0.0084,0.7889,0.0509,0.8542,0.0486,0.3121,0.0000
7,80,Trainable Quantum,0.9783,0.0114,0.8000,0.0333,0.8529,0.0320,0.5650,39.2520
8,160,Classical RBF,0.9836,0.0053,0.8500,0.0000,0.9157,0.0046,0.0409,0.0000
9,160,Fixed Reuploading,0.9746,0.0028,0.8444,0.0347,0.8849,0.0117,1.6417,0.0000


Optimization Stability Diagnostics

In [24]:

optimization_summary = (
    optimization_df
    .groupby("n_samples")
    .agg(
        alignment_gain_mean=("alignment_gain", "mean"),
        alignment_gain_std=("alignment_gain", "std"),
        optimization_time_mean=(
            "optimization_seconds", "mean"
        ),
        success_rate=(
            "optimization_success", "mean"
        ),
        mean_parameters_at_bound=(
            "parameters_at_lower_bound", "mean"
        )
    )
    .reset_index()
)

display(optimization_summary.round(4))

optimization_summary.to_csv(
    "phase24_optimization_summary.csv",
    index=False
)


,n_samples,alignment_gain_mean,alignment_gain_std,optimization_time_mean,success_rate,mean_parameters_at_bound
0,40,0.1302,0.0328,10.3277,1.0,1.6667
1,80,0.1374,0.0255,13.0840,1.0,2.4444
2,160,0.1179,0.0172,25.4668,1.0,2.5556


Feature Efficiency Setup

In [25]:

%pip install -q scikit-learn pandas numpy matplotlib


In [26]:

import numpy as np
import pandas as pd
import time

from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif
)
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, recall_score,
    precision_score, f1_score
)

TRAIN_SIZES = [40, 80, 160]
SEEDS = [42, 123, 2026]
N_FEATURES = 4

dataset = load_breast_cancer()

X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

feature_names = np.array(dataset.feature_names)

print("Dataset:", X_raw.shape)
print("Malignant:", sum(y_all == 1))
print("Benign:", sum(y_all == 0))


Dataset: (569, 30)
Malignant: 212
Benign: 357


Load Shared Training Indices

In [27]:

from google.colab import files
import zipfile

if not Path(
    "limited_data/data_n40_seed42.npz"
).exists():

    uploaded = files.upload()

    zip_name = next(
        name for name in uploaded
        if name.endswith(".zip")
    )

    with zipfile.ZipFile(zip_name) as z:
        z.extractall(".")

print("Shared training files ready.")


Shared training files ready.


Feature Selection Pipelines

In [28]:

def make_feature_pipeline(method, seed=42):

    if method == "PCA":
        selector = PCA(n_components=4)

    elif method == "ANOVA":
        selector = SelectKBest(
            score_func=f_classif,
            k=4
        )

    elif method == "Mutual Information":

        def mi_score(X, y):
            return mutual_info_classif(
                X, y,
                random_state=seed
            )

        selector = SelectKBest(
            score_func=mi_score,
            k=4
        )

    else:
        raise ValueError("Unknown method")

    return Pipeline([
        ("standardize", StandardScaler()),
        ("selector", selector),
        ("angles", MinMaxScaler(
            feature_range=(0, np.pi),
            clip=True
        )),
        ("svm", SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        ))
    ])


Compare Four-Feature Representations

In [29]:

METHODS = [
    "PCA",
    "ANOVA",
    "Mutual Information"
]

records = []

for n_samples in TRAIN_SIZES:
    for seed in SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            subset_idx = d["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for method in METHODS:

            for fold, (tr, val) in enumerate(
                cv.split(X_subset, y_subset)
            ):

                model = make_feature_pipeline(
                    method, seed
                )

                start = time.perf_counter()

                model.fit(
                    X_subset[tr],
                    y_subset[tr]
                )

                pred = model.predict(
                    X_subset[val]
                )

                scores = model.decision_function(
                    X_subset[val]
                )

                records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "method": method,
                    "roc_auc": roc_auc_score(
                        y_subset[val], scores
                    ),
                    "recall": recall_score(
                        y_subset[val], pred
                    ),
                    "precision": precision_score(
                        y_subset[val], pred,
                        zero_division=0
                    ),
                    "f1": f1_score(
                        y_subset[val], pred,
                        zero_division=0
                    ),
                    "runtime_seconds": (
                        time.perf_counter() - start
                    )
                })

            print(
                f"Completed n={n_samples}, "
                f"seed={seed}, method={method}"
            )

feature_cv_df = pd.DataFrame(records)

display(feature_cv_df.round(4))


Completed n=40, seed=42, method=PCA
Completed n=40, seed=42, method=ANOVA
Completed n=40, seed=42, method=Mutual Information
Completed n=40, seed=123, method=PCA
Completed n=40, seed=123, method=ANOVA
Completed n=40, seed=123, method=Mutual Information
Completed n=40, seed=2026, method=PCA
Completed n=40, seed=2026, method=ANOVA
Completed n=40, seed=2026, method=Mutual Information
Completed n=80, seed=42, method=PCA
Completed n=80, seed=42, method=ANOVA
Completed n=80, seed=42, method=Mutual Information
Completed n=80, seed=123, method=PCA
Completed n=80, seed=123, method=ANOVA
Completed n=80, seed=123, method=Mutual Information
Completed n=80, seed=2026, method=PCA
Completed n=80, seed=2026, method=ANOVA
Completed n=80, seed=2026, method=Mutual Information
Completed n=160, seed=42, method=PCA
Completed n=160, seed=42, method=ANOVA
Completed n=160, seed=42, method=Mutual Information
Completed n=160, seed=123, method=PCA
Completed n=160, seed=123, method=ANOVA
Completed n=160, seed=123,

,n_samples,seed,fold,method,roc_auc,recall,precision,f1,runtime_seconds
0,40,42,0,PCA,1.0000,0.80,1.0000,0.8889,0.0409
1,40,42,1,PCA,0.9500,0.60,1.0000,0.7500,0.0285
2,40,42,2,PCA,1.0000,1.00,1.0000,1.0000,0.0547
3,40,42,0,ANOVA,0.9556,0.60,0.7500,0.6667,0.0287
4,40,42,1,ANOVA,0.9500,0.80,0.8000,0.8000,0.0227
...,...,...,...,...,...,...,...,...,...
76,160,2026,1,ANOVA,0.9879,0.95,0.8261,0.8837,0.0346
77,160,2026,2,ANOVA,0.9894,0.85,0.9444,0.8947,0.0231
78,160,2026,0,Mutual Information,0.9794,0.90,0.9474,0.9231,0.1383
79,160,2026,1,Mutual Information,0.9894,0.95,0.7917,0.8636,0.1299


Feature Selection Performance Summary

In [30]:

per_seed = (
    feature_cv_df
    .groupby(["n_samples", "seed", "method"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        runtime=("runtime_seconds", "sum")
    )
    .reset_index()
)

feature_summary = (
    per_seed
    .groupby(["n_samples", "method"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(feature_summary.round(4))


,n_samples,method,auc_mean,auc_std,recall_mean,recall_std,f1_mean,f1_std,runtime_mean
0,40,ANOVA,0.9790,0.0200,0.8000,0.0000,0.8246,0.0503,0.0721
1,40,Mutual Information,0.9762,0.0241,0.8000,0.0000,0.8322,0.0158,0.6036
2,40,PCA,0.9549,0.0263,0.8000,0.0667,0.8305,0.0453,0.0931
3,80,ANOVA,0.9609,0.0277,0.8222,0.0839,0.8679,0.0612,0.1346
4,80,Mutual Information,0.9664,0.0203,0.8000,0.0577,0.8488,0.0432,0.8315
5,80,PCA,0.9834,0.0113,0.8111,0.0509,0.8738,0.0477,0.1282
6,160,ANOVA,0.9710,0.0181,0.8611,0.0255,0.8874,0.0104,0.0789
7,160,Mutual Information,0.9707,0.0139,0.8389,0.0536,0.8638,0.0281,0.3475
8,160,PCA,0.9848,0.0043,0.8833,0.0289,0.9187,0.0281,0.0729


Identify Selected Measurements

In [31]:

with np.load(
    "limited_data/data_n160_seed42.npz"
) as d:
    subset_idx = d["train_idx"].copy()

X_subset = X_raw[subset_idx]
y_subset = y_all[subset_idx]

selected_features = {}

for method in ["ANOVA", "Mutual Information"]:

    model = make_feature_pipeline(
        method, seed=42
    )

    model.fit(X_subset, y_subset)

    selector = model.named_steps["selector"]

    indices = selector.get_support(indices=True)

    selected_features[method] = (
        feature_names[indices].tolist()
    )

    print(f"\n{method}:")
    for name in feature_names[indices]:
        print(" -", name)



ANOVA:
 - mean concave points
 - worst radius
 - worst perimeter
 - worst concave points

Mutual Information:
 - mean concave points
 - worst radius
 - worst perimeter
 - worst concave points


Save Feature Selection Results

In [32]:

import json

output = Path("phase_3_results")
output.mkdir(exist_ok=True)

feature_cv_df.to_csv(
    output / "classical_feature_cv.csv",
    index=False
)

feature_summary.to_csv(
    output / "classical_feature_summary.csv",
    index=False
)

with open(
    output / "selected_features.json",
    "w"
) as f:
    json.dump(
        selected_features,
        f,
        indent=2
    )

print("Phase 3.1 results saved.")


Phase 3.1 results saved.


Quantum-Guided Feature Selection Setup

In [33]:

import numpy as np
import pandas as pd
import time

from pathlib import Path
from itertools import combinations

from sklearn.feature_selection import f_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, recall_score, f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC

import pennylane as qml

P32_SIZES = [40, 80]
P32_SEEDS = [42, 123, 2026]

N_CANDIDATES = 10
N_SELECTED = 4
C_FIXED = 1.0

P32_DIR = Path("phase_3_results")
P32_DIR.mkdir(exist_ok=True)

print("Phase 3.2 setup ready.")


Phase 3.2 setup ready.


Quantum Kernel for Feature Subsets

In [34]:

N_QUBITS = 4

dev_p32 = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(dev_p32)
def p32_quantum_state(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)
    return qml.state()


def p32_quantum_kernel(Xa, Xb):
    def pad(X):
        result = np.zeros((len(X), N_QUBITS))
        result[:, :X.shape[1]] = X
        return result

    Xa = pad(Xa)
    Xb = pad(Xb)

    states_a = np.asarray([
        p32_quantum_state(x) for x in Xa
    ], dtype=np.complex128)

    states_b = np.asarray([
        p32_quantum_state(x) for x in Xb
    ], dtype=np.complex128)

    return np.abs(
        states_a @ states_b.conj().T
    ) ** 2


Efficient Fidelity Kernel

In [35]:

def p32_fast_quantum_kernel(Xa, Xb):
    differences = (
        Xa[:, None, :] - Xb[None, :, :]
    )

    return np.prod(
        np.cos(differences / 2) ** 2,
        axis=2
    )


# Verify against PennyLane
rng = np.random.default_rng(42)
X_check = rng.uniform(
    0, np.pi, size=(8, 4)
)

K_sim = p32_quantum_kernel(
    X_check, X_check
)

K_fast = p32_fast_quantum_kernel(
    X_check, X_check
)

print(
    "Maximum kernel difference:",
    np.max(np.abs(K_sim - K_fast))
)

assert np.allclose(
    K_sim, K_fast,
    atol=1e-10
)

print("Fast kernel matches quantum simulation.")


Maximum kernel difference: 6.661338147750939e-16
Fast kernel matches quantum simulation.


Evaluate Candidate Feature Subsets

In [36]:

def p32_transform(Xtr, Xval, features):
    scaler = StandardScaler()
    angle_scaler = MinMaxScaler(
        feature_range=(0, np.pi),
        clip=True
    )

    A = scaler.fit_transform(
        Xtr[:, features]
    )

    B = scaler.transform(
        Xval[:, features]
    )

    A = angle_scaler.fit_transform(A)
    B = angle_scaler.transform(B)

    return A, B


def p32_candidate_auc(
    X, y, features, kernel_type, seed
):
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    aucs = []

    for tr, val in cv.split(X, y):

        A, B = p32_transform(
            X[tr], X[val], features
        )

        if kernel_type == "quantum":
            Ktr = p32_fast_quantum_kernel(A, A)
            Kval = p32_fast_quantum_kernel(B, A)

        else:
            gamma = 1.0 / (
                A.shape[1] * A.var()
            )

            Ktr = rbf_kernel(
                A, gamma=gamma
            )

            Kval = rbf_kernel(
                B, A, gamma=gamma
            )

        model = SVC(
            kernel="precomputed",
            C=C_FIXED
        )

        model.fit(Ktr, y[tr])

        scores = model.decision_function(Kval)

        aucs.append(
            roc_auc_score(y[val], scores)
        )

    return np.mean(aucs)


Select Four Features

In [37]:

def p32_greedy_select(
    X, y, candidates, kernel_type, seed
):
    selected = []
    history = []

    for step in range(N_SELECTED):

        best_feature = None
        best_score = -np.inf

        for feature in candidates:

            if feature in selected:
                continue

            trial = selected + [int(feature)]

            score = p32_candidate_auc(
                X, y, trial,
                kernel_type, seed
            )

            if score > best_score:
                best_score = score
                best_feature = int(feature)

        selected.append(best_feature)

        history.append({
            "step": step + 1,
            "feature": feature_names[best_feature],
            "cv_auc": best_score
        })

    return selected, history


Nested Classical vs Quantum Feature Selection

In [38]:

p32_records = []
p32_selected = []

for n_samples in P32_SIZES:
    for seed in P32_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            subset_idx = d["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        outer_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            outer_cv.split(X_subset, y_subset)
        ):

            Xtr = X_subset[tr]
            Xval = X_subset[val]

            ytr = y_subset[tr]
            yval = y_subset[val]

            # Candidate filter uses outer training only
            scores, _ = f_classif(Xtr, ytr)

            scores = np.nan_to_num(
                scores,
                nan=-np.inf
            )

            candidates = np.argsort(
                scores
            )[-N_CANDIDATES:][::-1]

            for kernel_type in [
                "classical", "quantum"
            ]:

                start = time.perf_counter()

                selected, history = p32_greedy_select(
                    Xtr, ytr,
                    candidates,
                    kernel_type,
                    seed + fold
                )

                A, B = p32_transform(
                    Xtr, Xval, selected
                )

                if kernel_type == "quantum":
                    Ktr = p32_fast_quantum_kernel(A, A)
                    Kval = p32_fast_quantum_kernel(B, A)

                else:
                    gamma = 1.0 / (
                        A.shape[1] * A.var()
                    )

                    Ktr = rbf_kernel(
                        A, gamma=gamma
                    )

                    Kval = rbf_kernel(
                        B, A, gamma=gamma
                    )

                model = SVC(
                    kernel="precomputed",
                    C=C_FIXED
                )

                model.fit(Ktr, ytr)

                pred = model.predict(Kval)
                decision = model.decision_function(Kval)

                p32_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "kernel": kernel_type,
                    "roc_auc": roc_auc_score(
                        yval, decision
                    ),
                    "recall": recall_score(
                        yval, pred,
                        zero_division=0
                    ),
                    "f1": f1_score(
                        yval, pred,
                        zero_division=0
                    ),
                    "runtime_seconds": (
                        time.perf_counter() - start
                    )
                })

                p32_selected.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "kernel": kernel_type,
                    "features": [
                        feature_names[i]
                        for i in selected
                    ],
                    "selection_history": history
                })

            print(
                f"Completed n={n_samples}, "
                f"seed={seed}, fold={fold}"
            )

        pd.DataFrame(p32_records).to_csv(
            P32_DIR / "greedy_cv.csv",
            index=False
        )

p32_df = pd.DataFrame(p32_records)
p32_features_df = pd.DataFrame(p32_selected)

print("Nested feature selection completed.")


Completed n=40, seed=42, fold=0
Completed n=40, seed=42, fold=1
Completed n=40, seed=42, fold=2
Completed n=40, seed=123, fold=0
Completed n=40, seed=123, fold=1
Completed n=40, seed=123, fold=2
Completed n=40, seed=2026, fold=0
Completed n=40, seed=2026, fold=1
Completed n=40, seed=2026, fold=2
Completed n=80, seed=42, fold=0
Completed n=80, seed=42, fold=1
Completed n=80, seed=42, fold=2
Completed n=80, seed=123, fold=0
Completed n=80, seed=123, fold=1
Completed n=80, seed=123, fold=2
Completed n=80, seed=2026, fold=0
Completed n=80, seed=2026, fold=1
Completed n=80, seed=2026, fold=2
Nested feature selection completed.


Feature Selection Results

In [39]:

p32_per_seed = (
    p32_df
    .groupby(["n_samples", "seed", "kernel"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        runtime=("runtime_seconds", "sum")
    )
    .reset_index()
)

p32_summary = (
    p32_per_seed
    .groupby(["n_samples", "kernel"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(p32_summary.round(4))

p32_summary.to_csv(
    P32_DIR / "greedy_summary.csv",
    index=False
)

p32_features_df.to_csv(
    P32_DIR / "greedy_selected_features.csv",
    index=False
)


,n_samples,kernel,auc_mean,auc_std,recall_mean,f1_mean,runtime_mean
0,40,classical,0.9790,0.0200,0.7778,0.8302,2.8281
1,40,quantum,0.9762,0.0241,0.7778,0.8352,2.3074
2,80,classical,0.9834,0.0149,0.8333,0.8680,4.3532
3,80,quantum,0.9868,0.0062,0.8000,0.8493,3.5949


Selected Feature Redundancy

In [40]:

def mean_feature_redundancy(X, selected):
    corr = np.corrcoef(
        X[:, selected],
        rowvar=False
    )

    upper = np.abs(
        corr[np.triu_indices_from(corr, k=1)]
    )

    return float(np.mean(upper))


redundancy_records = []

for row in p32_selected:

    path = Path(
        f"limited_data/data_n{row['n_samples']}_seed{row['seed']}.npz"
    )

    with np.load(path) as d:
        subset_idx = d["train_idx"].copy()

    selected_idx = [
        int(np.where(feature_names == name)[0][0])
        for name in row["features"]
    ]

    redundancy_records.append({
        "n_samples": row["n_samples"],
        "seed": row["seed"],
        "fold": row["fold"],
        "kernel": row["kernel"],
        "redundancy": mean_feature_redundancy(
            X_raw[subset_idx],
            selected_idx
        )
    })

redundancy_df = pd.DataFrame(
    redundancy_records
)

display(
    redundancy_df
    .groupby(["n_samples", "kernel"])
    ["redundancy"]
    .mean()
    .reset_index()
    .round(4)
)


,n_samples,kernel,redundancy
0,40,classical,0.8139
1,40,quantum,0.8184
2,80,classical,0.8202
3,80,quantum,0.8067


In [41]:

# 1. Check if classical and quantum selected
# different features

features = p32_features_df.copy()

features["feature_set"] = features["features"].apply(
    lambda x: tuple(sorted(x))
)

paired = features.pivot(
    index=["n_samples", "seed", "fold"],
    columns="kernel",
    values="feature_set"
).reset_index()

paired["same_features"] = (
    paired["classical"] == paired["quantum"]
)

print("FEATURE SELECTION AGREEMENT:")
display(
    paired.groupby("n_samples")["same_features"]
    .agg(["sum", "count", "mean"])
)

print("EXAMPLES OF DIFFERENT FEATURES:")
display(
    paired[~paired["same_features"]].head(5)
)

# 2. Calculate total experiment runtime

print("TOTAL RUNTIME BY METHOD (SECONDS):")
display(
    p32_df.groupby("kernel")["runtime_seconds"]
    .sum()
    .round(2)
)


FEATURE SELECTION AGREEMENT:


,sum,count,mean
n_samples,,,
40,7,9,0.777778
80,3,9,0.333333


EXAMPLES OF DIFFERENT FEATURES:


kernel,n_samples,seed,fold,classical,quantum,same_features
0,40,42,0,"(mean concave points, worst area, worst concav...","(mean concave points, mean perimeter, worst ar...",False
1,40,42,1,"(area error, worst area, worst concave points,...","(mean perimeter, worst area, worst concave poi...",False
10,80,42,1,"(mean compactness, worst area, worst concave p...","(mean compactness, worst area, worst perimeter...",False
11,80,42,2,"(mean concave points, mean concavity, worst co...","(worst area, worst concave points, worst conca...",False
12,80,123,0,"(worst area, worst concavity, worst perimeter,...","(mean concave points, worst compactness, worst...",False


TOTAL RUNTIME BY METHOD (SECONDS):


,runtime_seconds
kernel,
classical,21.54
quantum,17.71


Install and import libraries

In [42]:

import numpy as np
import pandas as pd
import pennylane as qml
import time
import zipfile

from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

from sklearn.metrics.pairwise import rbf_kernel

TRAIN_SIZES = [40, 80, 160]
SEEDS = [42, 123, 2026]

NOISE_LEVELS = [0.00, 0.02, 0.05, 0.10, 0.20]
NOISE_REPEATS = 3

N_QUBITS = 4
C_FIXED = 1.0

OUTPUT_DIR = Path("phase_4_results")
OUTPUT_DIR.mkdir(exist_ok=True)

dataset = load_breast_cancer()

X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

print("Phase 4 initialized.")


Phase 4 initialized.


In [43]:

from google.colab import files

if not Path(
    "limited_data/data_n40_seed42.npz"
).exists():

    uploaded = files.upload()

    zip_name = next(
        name for name in uploaded
        if name.endswith(".zip")
    )

    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(".")

print("Training subsets ready.")


Training subsets ready.


Quantum Feature Maps

In [44]:

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS
)

@qml.qnode(dev)
def simple_ry_state(x):
    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)
    return qml.state()


@qml.qnode(dev)
def reupload_state(x):

    for i in range(N_QUBITS):
        qml.RY(x[i], wires=i)

    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    for i in range(N_QUBITS):
        qml.RZ(x[i], wires=i)
        qml.RY(x[i], wires=i)

    return qml.state()


def quantum_states(X, state_fn):
    return np.asarray(
        [state_fn(x) for x in X],
        dtype=np.complex128
    )


def fidelity_kernel(A, B):
    return np.abs(
        A @ B.conj().T
    ) ** 2


Generate Noisy Diagnostic Measurements

In [45]:

def prepare_noisy_features(
    X_train_raw,
    X_val_raw,
    noise_level,
    rng
):
    # Fit only on clean training data
    scaler = StandardScaler()
    pca = PCA(n_components=4)

    angle_scaler = MinMaxScaler(
        feature_range=(0, np.pi),
        clip=True
    )

    train_standardized = scaler.fit_transform(
        X_train_raw
    )

    val_standardized = scaler.transform(
        X_val_raw
    )

    # Synthetic noise in standardized feature space
    noise = rng.normal(
        loc=0.0,
        scale=noise_level,
        size=val_standardized.shape
    )

    val_noisy = val_standardized + noise

    train_pca = pca.fit_transform(
        train_standardized
    )

    val_pca = pca.transform(
        val_noisy
    )

    X_train = angle_scaler.fit_transform(
        train_pca
    )

    X_val = angle_scaler.transform(
        val_pca
    )

    return X_train, X_val


Compute Classical and Quantum Kernels

In [46]:

def make_kernel_pair(
    X_train,
    X_val,
    model_name
):

    if model_name == "Classical RBF":
        gamma = 1.0 / (
            X_train.shape[1] * X_train.var()
        )

        K_train = rbf_kernel(
            X_train,
            gamma=gamma
        )

        K_val = rbf_kernel(
            X_val,
            X_train,
            gamma=gamma
        )

    else:
        state_fn = (
            simple_ry_state
            if model_name == "Simple RY"
            else reupload_state
        )

        states_train = quantum_states(
            X_train,
            state_fn
        )

        states_val = quantum_states(
            X_val,
            state_fn
        )

        K_train = fidelity_kernel(
            states_train,
            states_train
        )

        K_val = fidelity_kernel(
            states_val,
            states_train
        )

    return K_train, K_val


Multi-Seed Noise Robustness Benchmark

In [47]:

MODELS = [
    "Classical RBF",
    "Simple RY",
    "Data Reuploading"
]

records = []

for n_samples in TRAIN_SIZES:
    for seed in SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            cv.split(X_subset, y_subset)
        ):

            Xtr_raw = X_subset[tr]
            Xval_raw = X_subset[val]

            ytr = y_subset[tr]
            yval = y_subset[val]

            for noise_level in NOISE_LEVELS:
                for repeat in range(NOISE_REPEATS):

                    rng = np.random.default_rng(
                        seed * 10000
                        + fold * 1000
                        + repeat * 100
                        + int(noise_level * 100)
                    )

                    Xtr, Xval = prepare_noisy_features(
                        Xtr_raw,
                        Xval_raw,
                        noise_level,
                        rng
                    )

                    for model_name in MODELS:

                        start = time.perf_counter()

                        Ktr, Kval = make_kernel_pair(
                            Xtr,
                            Xval,
                            model_name
                        )

                        model = SVC(
                            kernel="precomputed",
                            C=C_FIXED
                        )

                        model.fit(Ktr, ytr)

                        predictions = model.predict(Kval)
                        scores = model.decision_function(Kval)

                        tn, fp, fn, tp = confusion_matrix(
                            yval,
                            predictions,
                            labels=[0, 1]
                        ).ravel()

                        records.append({
                            "n_samples": n_samples,
                            "seed": seed,
                            "fold": fold,
                            "noise_level": noise_level,
                            "repeat": repeat,
                            "model": model_name,
                            "roc_auc": roc_auc_score(
                                yval, scores
                            ),
                            "recall": recall_score(
                                yval, predictions,
                                zero_division=0
                            ),
                            "precision": precision_score(
                                yval, predictions,
                                zero_division=0
                            ),
                            "f1": f1_score(
                                yval, predictions,
                                zero_division=0
                            ),
                            "false_negatives": int(fn),
                            "false_positives": int(fp),
                            "runtime_seconds": (
                                time.perf_counter() - start
                            )
                        })

            print(
                f"Completed n={n_samples}, "
                f"seed={seed}, fold={fold}"
            )

        # Save progress after each seed
        pd.DataFrame(records).to_csv(
            OUTPUT_DIR / "robustness_raw.csv",
            index=False
        )

robustness_df = pd.DataFrame(records)

print("Phase 4 experiments completed.")


Completed n=40, seed=42, fold=0
Completed n=40, seed=42, fold=1
Completed n=40, seed=42, fold=2
Completed n=40, seed=123, fold=0
Completed n=40, seed=123, fold=1
Completed n=40, seed=123, fold=2
Completed n=40, seed=2026, fold=0
Completed n=40, seed=2026, fold=1
Completed n=40, seed=2026, fold=2
Completed n=80, seed=42, fold=0
Completed n=80, seed=42, fold=1
Completed n=80, seed=42, fold=2
Completed n=80, seed=123, fold=0
Completed n=80, seed=123, fold=1
Completed n=80, seed=123, fold=2
Completed n=80, seed=2026, fold=0
Completed n=80, seed=2026, fold=1
Completed n=80, seed=2026, fold=2
Completed n=160, seed=42, fold=0
Completed n=160, seed=42, fold=1
Completed n=160, seed=42, fold=2
Completed n=160, seed=123, fold=0
Completed n=160, seed=123, fold=1
Completed n=160, seed=123, fold=2
Completed n=160, seed=2026, fold=0
Completed n=160, seed=2026, fold=1
Completed n=160, seed=2026, fold=2
Phase 4 experiments completed.


Summarize Robustness Across Noise Levels

In [48]:

# Average repeated noise draws and folds within each seed
per_seed = (
    robustness_df
    .groupby([
        "n_samples",
        "seed",
        "noise_level",
        "model"
    ])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean"),
        fp=("false_positives", "mean")
    )
    .reset_index()
)

# Aggregate across the three seeds
robustness_summary = (
    per_seed
    .groupby([
        "n_samples",
        "noise_level",
        "model"
    ])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        fp_mean=("fp", "mean")
    )
    .reset_index()
)

display(robustness_summary.round(4))

robustness_summary.to_csv(
    OUTPUT_DIR / "robustness_summary.csv",
    index=False
)


,n_samples,noise_level,model,auc_mean,auc_std,recall_mean,f1_mean,fp_mean
0,40,0.00,Classical RBF,0.9549,0.0263,0.8000,0.8305,0.5556
1,40,0.00,Data Reuploading,0.9552,0.0104,0.7778,0.8142,0.5556
2,40,0.00,Simple RY,0.9469,0.0333,0.8000,0.8305,0.5556
3,40,0.02,Classical RBF,0.9549,0.0263,0.7926,0.8268,0.5556
4,40,0.02,Data Reuploading,0.9562,0.0102,0.7778,0.8109,0.5926
5,40,0.02,Simple RY,0.9469,0.0333,0.8000,0.8305,0.5556
6,40,0.05,Classical RBF,0.9540,0.0275,0.7926,0.8268,0.5556
7,40,0.05,Data Reuploading,0.9562,0.0102,0.7926,0.8124,0.6667
8,40,0.05,Simple RY,0.9469,0.0333,0.8000,0.8305,0.5556
9,40,0.10,Classical RBF,0.9567,0.0304,0.7926,0.8242,0.5926


Quantify Robustness Degradation

In [49]:

clean = robustness_summary[
    robustness_summary["noise_level"] == 0
][[
    "n_samples",
    "model",
    "auc_mean",
    "recall_mean",
    "f1_mean"
]].rename(columns={
    "auc_mean": "clean_auc",
    "recall_mean": "clean_recall",
    "f1_mean": "clean_f1"
})

degradation_df = robustness_summary.merge(
    clean,
    on=["n_samples", "model"],
    validate="many_to_one"
)

degradation_df["auc_drop"] = (
    degradation_df["clean_auc"]
    - degradation_df["auc_mean"]
)

degradation_df["recall_drop"] = (
    degradation_df["clean_recall"]
    - degradation_df["recall_mean"]
)

degradation_df["f1_drop"] = (
    degradation_df["clean_f1"]
    - degradation_df["f1_mean"]
)

display(
    degradation_df[[
        "n_samples",
        "noise_level",
        "model",
        "auc_mean",
        "auc_drop",
        "recall_mean",
        "recall_drop",
        "f1_mean",
        "f1_drop"
    ]].round(4)
)

degradation_df.to_csv(
    OUTPUT_DIR / "robustness_degradation.csv",
    index=False
)


,n_samples,noise_level,model,auc_mean,auc_drop,recall_mean,recall_drop,f1_mean,f1_drop
0,40,0.00,Classical RBF,0.9549,0.0000,0.8000,0.0000,0.8305,0.0000
1,40,0.00,Data Reuploading,0.9552,0.0000,0.7778,0.0000,0.8142,0.0000
2,40,0.00,Simple RY,0.9469,0.0000,0.8000,0.0000,0.8305,0.0000
3,40,0.02,Classical RBF,0.9549,0.0000,0.7926,0.0074,0.8268,0.0037
4,40,0.02,Data Reuploading,0.9562,-0.0009,0.7778,0.0000,0.8109,0.0033
5,40,0.02,Simple RY,0.9469,0.0000,0.8000,0.0000,0.8305,0.0000
6,40,0.05,Classical RBF,0.9540,0.0009,0.7926,0.0074,0.8268,0.0037
7,40,0.05,Data Reuploading,0.9562,-0.0009,0.7926,-0.0148,0.8124,0.0018
8,40,0.05,Simple RY,0.9469,0.0000,0.8000,0.0000,0.8305,0.0000
9,40,0.10,Classical RBF,0.9567,-0.0017,0.7926,0.0074,0.8242,0.0064


Compare Models at 20% Noise

In [50]:

high_noise = degradation_df[
    degradation_df["noise_level"] == 0.20
]

high_noise_summary = high_noise[[
    "n_samples",
    "model",
    "auc_mean",
    "auc_drop",
    "recall_mean",
    "recall_drop",
    "f1_mean",
    "f1_drop"
]].sort_values(
    ["n_samples", "model"]
)

display(high_noise_summary.round(4))


,n_samples,model,auc_mean,auc_drop,recall_mean,recall_drop,f1_mean,f1_drop
12,40,Classical RBF,0.9546,0.0003,0.7852,0.0148,0.8156,0.0149
13,40,Data Reuploading,0.9496,0.0057,0.7704,0.0074,0.7957,0.0185
14,40,Simple RY,0.9450,0.0020,0.8000,-0.0000,0.8258,0.0047
27,80,Classical RBF,0.9835,-0.0000,0.8259,-0.0148,0.8803,-0.0065
28,80,Data Reuploading,0.9669,0.0006,0.8148,-0.0148,0.8519,-0.0075
29,80,Simple RY,0.9853,0.0027,0.7963,-0.0074,0.8622,-0.0080
42,160,Classical RBF,0.9844,0.0003,0.8741,0.0093,0.9171,0.0016
43,160,Data Reuploading,0.9731,0.0015,0.8426,0.0019,0.8889,-0.0040
44,160,Simple RY,0.9834,-0.0007,0.8722,-0.0056,0.9179,-0.0085


Kernel Mixture Setup

In [51]:

import numpy as np
import pandas as pd
import pennylane as qml
import time
import zipfile

from pathlib import Path
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score
)
from sklearn.svm import SVC

P5_SIZES = [40, 80, 160]
P5_SEEDS = [42, 123, 2026]
P5_WEIGHTS = [0, 0.25, 0.5, 0.75, 1]

P5_C = 1.0
P5_QUBITS = 4

P5_DIR = Path("phase_5_results")
P5_DIR.mkdir(exist_ok=True)

dataset = load_breast_cancer()
X_raw = dataset.data
y_all = (dataset.target == 0).astype(int)

print("Phase 5 initialized.")


Phase 5 initialized.


Load Reproducible Training Indices

In [52]:

from google.colab import files

if not Path(
    "limited_data/data_n40_seed42.npz"
).exists():
    uploaded = files.upload()
    zip_name = next(
        name for name in uploaded
        if name.endswith(".zip")
    )
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(".")

print("Shared training subsets available.")


Shared training subsets available.


Fixed Quantum Kernels

In [53]:

p5_dev = qml.device(
    "default.qubit",
    wires=P5_QUBITS
)

@qml.qnode(p5_dev)
def p5_simple_state(x):
    for i in range(P5_QUBITS):
        qml.RY(x[i], wires=i)
    return qml.state()


@qml.qnode(p5_dev)
def p5_reupload_state(x):
    for i in range(P5_QUBITS):
        qml.RY(x[i], wires=i)

    for i in range(P5_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])

    for i in range(P5_QUBITS):
        qml.RZ(x[i], wires=i)
        qml.RY(x[i], wires=i)

    return qml.state()


def p5_state_matrix(X, state_fn):
    return np.asarray(
        [state_fn(x) for x in X],
        dtype=np.complex128
    )


def p5_kernel_pair(Xtr, Xval, state_fn):
    A = p5_state_matrix(Xtr, state_fn)
    B = p5_state_matrix(Xval, state_fn)

    return (
        np.abs(A @ A.conj().T) ** 2,
        np.abs(B @ A.conj().T) ** 2
    )


Leakage-Safe Feature Transformation

In [54]:

def p5_transform(Xtr_raw, Xval_raw):
    scaler = StandardScaler()
    pca = PCA(n_components=4)
    angles = MinMaxScaler(
        feature_range=(0, np.pi),
        clip=True
    )

    Xtr = scaler.fit_transform(Xtr_raw)
    Xval = scaler.transform(Xval_raw)

    Xtr = pca.fit_transform(Xtr)
    Xval = pca.transform(Xval)

    Xtr = angles.fit_transform(Xtr)
    Xval = angles.transform(Xval)

    return Xtr, Xval


Build Kernel Mixtures

In [55]:

def p5_get_kernel_components(Xtr, Xval, kind):

    if kind == "quantum":
        K1_tr, K1_val = p5_kernel_pair(
            Xtr, Xval, p5_simple_state
        )
        K2_tr, K2_val = p5_kernel_pair(
            Xtr, Xval, p5_reupload_state
        )

    elif kind == "classical":
        K1_tr = rbf_kernel(
            Xtr, gamma=0.1
        )
        K1_val = rbf_kernel(
            Xval, Xtr, gamma=0.1
        )

        K2_tr = rbf_kernel(
            Xtr, gamma=1.0
        )
        K2_val = rbf_kernel(
            Xval, Xtr, gamma=1.0
        )

    else:
        raise ValueError("Unknown kernel kind")

    return K1_tr, K1_val, K2_tr, K2_val


def p5_mix_kernels(
    K1_tr, K1_val,
    K2_tr, K2_val,
    alpha
):
    Ktr = (
        alpha * K1_tr +
        (1 - alpha) * K2_tr
    )

    Kval = (
        alpha * K1_val +
        (1 - alpha) * K2_val
    )

    return Ktr, Kval


Multi-Seed Kernel Mixture Evaluation

In [56]:

p5_records = []

for n_samples in P5_SIZES:
    for seed in P5_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as d:
            subset_idx = d["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            cv.split(X_subset, y_subset)
        ):

            Xtr, Xval = p5_transform(
                X_subset[tr],
                X_subset[val]
            )

            ytr = y_subset[tr]
            yval = y_subset[val]

            for kind in ["classical", "quantum"]:

                start = time.perf_counter()

                components = p5_get_kernel_components(
                    Xtr, Xval, kind
                )

                component_seconds = (
                    time.perf_counter() - start
                )

                for alpha in P5_WEIGHTS:

                    Ktr, Kval = p5_mix_kernels(
                        *components, alpha
                    )

                    model = SVC(
                        kernel="precomputed",
                        C=P5_C
                    )

                    model.fit(Ktr, ytr)

                    pred = model.predict(Kval)
                    scores = model.decision_function(Kval)

                    p5_records.append({
                        "n_samples": n_samples,
                        "seed": seed,
                        "fold": fold,
                        "kind": kind,
                        "alpha": alpha,
                        "roc_auc": roc_auc_score(
                            yval, scores
                        ),
                        "recall": recall_score(
                            yval, pred,
                            zero_division=0
                        ),
                        "f1": f1_score(
                            yval, pred,
                            zero_division=0
                        ),
                        "kernel_seconds": (
                            component_seconds
                        )
                    })

            print(
                f"Completed n={n_samples}, "
                f"seed={seed}, fold={fold}"
            )

        pd.DataFrame(p5_records).to_csv(
            P5_DIR / "mixture_ablation_raw.csv",
            index=False
        )

p5_df = pd.DataFrame(p5_records)

print("Kernel mixture ablation complete.")


Completed n=40, seed=42, fold=0
Completed n=40, seed=42, fold=1
Completed n=40, seed=42, fold=2
Completed n=40, seed=123, fold=0
Completed n=40, seed=123, fold=1
Completed n=40, seed=123, fold=2
Completed n=40, seed=2026, fold=0
Completed n=40, seed=2026, fold=1
Completed n=40, seed=2026, fold=2
Completed n=80, seed=42, fold=0
Completed n=80, seed=42, fold=1
Completed n=80, seed=42, fold=2
Completed n=80, seed=123, fold=0
Completed n=80, seed=123, fold=1
Completed n=80, seed=123, fold=2
Completed n=80, seed=2026, fold=0
Completed n=80, seed=2026, fold=1
Completed n=80, seed=2026, fold=2
Completed n=160, seed=42, fold=0
Completed n=160, seed=42, fold=1
Completed n=160, seed=42, fold=2
Completed n=160, seed=123, fold=0
Completed n=160, seed=123, fold=1
Completed n=160, seed=123, fold=2
Completed n=160, seed=2026, fold=0
Completed n=160, seed=2026, fold=1
Completed n=160, seed=2026, fold=2
Kernel mixture ablation complete.


Compare Mixture Weights

In [57]:

p5_per_seed = (
    p5_df
    .groupby([
        "n_samples", "seed", "kind", "alpha"
    ])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        f1=("f1", "mean")
    )
    .reset_index()
)

p5_summary = (
    p5_per_seed
    .groupby([
        "n_samples", "kind", "alpha"
    ])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean")
    )
    .reset_index()
)

display(p5_summary.round(4))

p5_summary.to_csv(
    P5_DIR / "mixture_ablation_summary.csv",
    index=False
)


,n_samples,kind,alpha,auc_mean,auc_std,recall_mean,f1_mean
0,40,classical,0.00,0.9512,0.0285,0.8222,0.8342
1,40,classical,0.25,0.9543,0.0312,0.8000,0.8232
2,40,classical,0.50,0.9651,0.0159,0.8222,0.8426
3,40,classical,0.75,0.9682,0.0133,0.7556,0.7988
4,40,classical,1.00,0.9633,0.0282,0.6667,0.7438
5,40,quantum,0.00,0.9552,0.0104,0.7778,0.8142
6,40,quantum,0.25,0.9682,0.0194,0.8222,0.8426
7,40,quantum,0.50,0.9654,0.0185,0.8222,0.8426
8,40,quantum,0.75,0.9738,0.0156,0.8222,0.8426
9,40,quantum,1.00,0.9469,0.0333,0.8000,0.8305


Mixture vs Single-Kernel Controls

In [58]:

mixture_findings = []

for (n_samples, kind), group in p5_summary.groupby(
    ["n_samples", "kind"]
):

    endpoints = group[
        group["alpha"].isin([0, 1])
    ]

    mixtures = group[
        ~group["alpha"].isin([0, 1])
    ]

    best_endpoint_auc = endpoints["auc_mean"].max()
    best_mixture_row = mixtures.loc[
        mixtures["auc_mean"].idxmax()
    ]

    mixture_findings.append({
        "n_samples": n_samples,
        "kind": kind,
        "best_endpoint_auc": best_endpoint_auc,
        "best_mixture_alpha": best_mixture_row["alpha"],
        "best_mixture_auc": best_mixture_row["auc_mean"],
        "auc_difference": (
            best_mixture_row["auc_mean"]
            - best_endpoint_auc
        )
    })

mixture_findings_df = pd.DataFrame(
    mixture_findings
)

display(mixture_findings_df.round(4))


,n_samples,kind,best_endpoint_auc,best_mixture_alpha,best_mixture_auc,auc_difference
0,40,classical,0.9633,0.75,0.9682,0.0049
1,40,quantum,0.9552,0.75,0.9738,0.0185
2,80,classical,0.9878,0.75,0.9841,-0.0037
3,80,quantum,0.9880,0.75,0.9894,0.0013
4,160,classical,0.9836,0.50,0.9849,0.0014
5,160,quantum,0.9827,0.75,0.9841,0.0013


Reproducibility and Kernel Validation

In [59]:

# Quick PSD check on one representative training subset
with np.load(
    "limited_data/data_n80_seed42.npz"
) as d:
    X_check = d["X_train"].copy()

K1, _, K2, _ = p5_get_kernel_components(
    X_check, X_check, "quantum"
)

for alpha in P5_WEIGHTS:
    K = alpha * K1 + (1 - alpha) * K2

    assert np.allclose(K, K.T, atol=1e-8)
    assert np.allclose(np.diag(K), 1, atol=1e-8)

    min_eigenvalue = np.linalg.eigvalsh(K).min()

    assert min_eigenvalue > -1e-8

    print(
        f"alpha={alpha:.2f}, "
        f"min eigenvalue={min_eigenvalue:.8f}"
    )

mixture_findings_df.to_csv(
    P5_DIR / "mixture_vs_endpoints.csv",
    index=False
)

print("Phase 5 results saved and kernels validated.")


alpha=0.00, min eigenvalue=0.00000113
alpha=0.25, min eigenvalue=0.00000150
alpha=0.50, min eigenvalue=0.00000120
alpha=0.75, min eigenvalue=0.00000071
alpha=1.00, min eigenvalue=0.00000000
Phase 5 results saved and kernels validated.


Nested Kernel Mixture Setup

In [60]:

import numpy as np
import pandas as pd
import time

from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

P510_SIZES = [40, 80]
P510_SEEDS = [42, 123, 2026]
P510_WEIGHTS = [0, 0.25, 0.5, 0.75, 1.0]

P510_C = 1.0

P510_DIR = Path("phase_5_results")
P510_DIR.mkdir(exist_ok=True)

p510_records = []
p510_selection_records = []

print("Nested kernel mixture experiment ready.")


Nested kernel mixture experiment ready.


Select Mixture Weight Using Training Data Only

In [61]:

def p510_select_alpha(
    X_train_raw,
    y_train,
    kind,
    seed
):
    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    scores_by_alpha = {
        alpha: []
        for alpha in P510_WEIGHTS
    }

    for inner_tr, inner_val in inner_cv.split(
        X_train_raw, y_train
    ):

        Xtr, Xval = p5_transform(
            X_train_raw[inner_tr],
            X_train_raw[inner_val]
        )

        ytr = y_train[inner_tr]
        yval = y_train[inner_val]

        components = p5_get_kernel_components(
            Xtr,
            Xval,
            kind
        )

        for alpha in P510_WEIGHTS:

            Ktr, Kval = p5_mix_kernels(
                *components,
                alpha
            )

            model = SVC(
                kernel="precomputed",
                C=P510_C
            )

            model.fit(Ktr, ytr)

            scores = model.decision_function(Kval)

            auc = roc_auc_score(
                yval,
                scores
            )

            scores_by_alpha[alpha].append(auc)

    mean_scores = {
        alpha: float(np.mean(values))
        for alpha, values in scores_by_alpha.items()
    }

    # Deterministic tie-breaking: first alpha
    # in the predefined candidate list wins.
    best_alpha = max(
        P510_WEIGHTS,
        key=lambda a: mean_scores[a]
    )

    return best_alpha, mean_scores


Outer-Fold Evaluation

In [62]:

def p510_evaluate(
    X_train_raw,
    y_train,
    X_val_raw,
    y_val,
    kind,
    alpha
):
    Xtr, Xval = p5_transform(
        X_train_raw,
        X_val_raw
    )

    components = p5_get_kernel_components(
        Xtr,
        Xval,
        kind
    )

    Ktr, Kval = p5_mix_kernels(
        *components,
        alpha
    )

    model = SVC(
        kernel="precomputed",
        C=P510_C
    )

    model.fit(Ktr, y_train)

    pred = model.predict(Kval)
    scores = model.decision_function(Kval)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        pred,
        labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(y_val, scores),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        ),
        "false_negatives": int(fn),
        "false_positives": int(fp)
    }


Nested Classical vs Quantum Evaluation

In [63]:

p510_records = []
p510_selection_records = []

for n_samples in P510_SIZES:
    for seed in P510_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        outer_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (outer_tr, outer_val) in enumerate(
            outer_cv.split(X_subset, y_subset)
        ):

            Xtr_raw = X_subset[outer_tr]
            Xval_raw = X_subset[outer_val]

            ytr = y_subset[outer_tr]
            yval = y_subset[outer_val]

            for kind in ["classical", "quantum"]:

                start = time.perf_counter()

                # Inner training-only selection
                best_alpha, inner_scores = (
                    p510_select_alpha(
                        Xtr_raw,
                        ytr,
                        kind,
                        seed + fold
                    )
                )

                # Outer validation
                metrics = p510_evaluate(
                    Xtr_raw,
                    ytr,
                    Xval_raw,
                    yval,
                    kind,
                    best_alpha
                )

                elapsed = (
                    time.perf_counter() - start
                )

                p510_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "kind": kind,
                    "selected_alpha": best_alpha,
                    "runtime_seconds": elapsed,
                    **metrics
                })

                for alpha, auc in inner_scores.items():
                    p510_selection_records.append({
                        "n_samples": n_samples,
                        "seed": seed,
                        "fold": fold,
                        "kind": kind,
                        "alpha": alpha,
                        "inner_cv_auc": auc,
                        "selected": (
                            alpha == best_alpha
                        )
                    })

                print(
                    f"n={n_samples}, seed={seed}, "
                    f"fold={fold}, {kind}: "
                    f"alpha={best_alpha}, "
                    f"AUC={metrics['roc_auc']:.4f}"
                )

        # Save progress after each seed
        pd.DataFrame(p510_records).to_csv(
            P510_DIR / "nested_mixture_results.csv",
            index=False
        )

        pd.DataFrame(
            p510_selection_records
        ).to_csv(
            P510_DIR / "nested_mixture_selection.csv",
            index=False
        )

p510_df = pd.DataFrame(p510_records)

print("Nested mixture validation complete.")


n=40, seed=42, fold=0, classical: alpha=1.0, AUC=1.0000
n=40, seed=42, fold=0, quantum: alpha=1.0, AUC=1.0000
n=40, seed=42, fold=1, classical: alpha=0.5, AUC=0.9500
n=40, seed=42, fold=1, quantum: alpha=1.0, AUC=0.9250
n=40, seed=42, fold=2, classical: alpha=0, AUC=1.0000
n=40, seed=42, fold=2, quantum: alpha=0.25, AUC=1.0000
n=40, seed=123, fold=0, classical: alpha=0, AUC=1.0000
n=40, seed=123, fold=0, quantum: alpha=0.25, AUC=1.0000
n=40, seed=123, fold=1, classical: alpha=0, AUC=0.8250
n=40, seed=123, fold=1, quantum: alpha=0, AUC=0.9000
n=40, seed=123, fold=2, classical: alpha=0, AUC=1.0000
n=40, seed=123, fold=2, quantum: alpha=0.75, AUC=1.0000
n=40, seed=2026, fold=0, classical: alpha=0, AUC=0.9111
n=40, seed=2026, fold=0, quantum: alpha=0.5, AUC=0.8889
n=40, seed=2026, fold=1, classical: alpha=0, AUC=0.8750
n=40, seed=2026, fold=1, quantum: alpha=0, AUC=0.9500
n=40, seed=2026, fold=2, classical: alpha=1.0, AUC=1.0000
n=40, seed=2026, fold=2, quantum: alpha=0.25, AUC=1.0000
n=80

Aggregate Nested Validation Results

In [64]:

p510_per_seed = (
    p510_df
    .groupby(["n_samples", "seed", "kind"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        runtime=("runtime_seconds", "sum")
    )
    .reset_index()
)

p510_summary = (
    p510_per_seed
    .groupby(["n_samples", "kind"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(p510_summary.round(4))

p510_per_seed.to_csv(
    P510_DIR / "nested_mixture_per_seed.csv",
    index=False
)

p510_summary.to_csv(
    P510_DIR / "nested_mixture_summary.csv",
    index=False
)


,n_samples,kind,auc_mean,auc_std,recall_mean,recall_std,precision_mean,f1_mean,runtime_mean
0,40,classical,0.9512,0.0285,0.7778,0.0770,0.9189,0.8064,0.4085
1,40,quantum,0.9627,0.0148,0.8222,0.0385,0.9398,0.8487,3.0120
2,80,classical,0.9866,0.0057,0.8222,0.0694,0.9677,0.8802,0.3614
3,80,quantum,0.9835,0.0099,0.8222,0.0509,0.9308,0.8665,5.7236


Mixture-Weight Selection Stability

In [65]:

alpha_counts = (
    p510_df
    .groupby([
        "n_samples",
        "kind",
        "selected_alpha"
    ])
    .size()
    .reset_index(name="count")
)

display(alpha_counts)

alpha_counts.to_csv(
    P510_DIR / "selected_alpha_counts.csv",
    index=False
)


,n_samples,kind,selected_alpha,count
0,40,classical,0.00,6
1,40,classical,0.50,1
2,40,classical,1.00,2
3,40,quantum,0.00,2
4,40,quantum,0.25,3
5,40,quantum,0.50,1
6,40,quantum,0.75,1
7,40,quantum,1.00,2
8,80,classical,0.25,2
9,80,classical,0.50,2


Paired Quantum vs Classical Differences

In [66]:

paired = p510_df.pivot(
    index=["n_samples", "seed", "fold"],
    columns="kind",
    values=["roc_auc", "recall", "f1"]
)

paired.columns = [
    f"{metric}_{kind}"
    for metric, kind in paired.columns
]

paired = paired.reset_index()

for metric in ["roc_auc", "recall", "f1"]:
    paired[f"{metric}_difference"] = (
        paired[f"{metric}_quantum"]
        - paired[f"{metric}_classical"]
    )

paired_summary = (
    paired
    .groupby("n_samples")
    [[
        "roc_auc_difference",
        "recall_difference",
        "f1_difference"
    ]]
    .agg(["mean", "std"])
)

display(paired_summary.round(4))

paired.to_csv(
    P510_DIR / "paired_mixture_differences.csv",
    index=False
)


roc_auc_difference         recall_difference         f1_difference  \
                        mean     std              mean     std          mean   
n_samples                                                                      
40                    0.0114  0.0374            0.0444  0.0882        0.0423   
80                   -0.0031  0.0130           -0.0000  0.0866       -0.0137   

                   
              std  
n_samples          
40         0.0551  
80         0.0587

Strong Classical Baseline and Quantum Ablation

In [67]:

import numpy as np
import pandas as pd
import time
import json

from pathlib import Path
from itertools import product

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)
from sklearn.metrics.pairwise import rbf_kernel

P511_SIZES = [40, 80]
P511_SEEDS = [42, 123, 2026]

P511_WEIGHTS = [0, 0.25, 0.5, 0.75, 1.0]
P511_C_VALUES = [0.1, 1, 10, 100]
P511_GAMMAS = ["scale", 0.01, 0.1, 1.0]

P511_DIR = Path("phase_5_results")
P511_DIR.mkdir(exist_ok=True)

P511_MODELS = [
    "Quantum Mixture",
    "Quantum Endpoint",
    "Classical Mixture",
    "Tuned Classical RBF"
]

print("Phase 5.11 ready.")


Phase 5.11 ready.


Candidate Configurations

In [68]:

def p511_configs(model_name):

    if model_name == "Quantum Mixture":
        return [
            {"alpha": a, "C": 1.0}
            for a in P511_WEIGHTS
        ]

    if model_name == "Quantum Endpoint":
        return [
            {"alpha": a, "C": 1.0}
            for a in [0.0, 1.0]
        ]

    if model_name == "Classical Mixture":
        return [
            {"alpha": a, "C": 1.0}
            for a in P511_WEIGHTS
        ]

    if model_name == "Tuned Classical RBF":
        return [
            {"C": C, "gamma": gamma}
            for C, gamma in product(
                P511_C_VALUES,
                P511_GAMMAS
            )
        ]

    raise ValueError("Unknown model")


Build Model Kernels

In [69]:

def p511_kernel_pair(
    Xtr, Xval, model_name, config
):

    if model_name in [
        "Quantum Mixture",
        "Quantum Endpoint"
    ]:
        components = p5_get_kernel_components(
            Xtr, Xval, "quantum"
        )

        return p5_mix_kernels(
            *components,
            config["alpha"]
        )

    if model_name == "Classical Mixture":
        components = p5_get_kernel_components(
            Xtr, Xval, "classical"
        )

        return p5_mix_kernels(
            *components,
            config["alpha"]
        )

    if model_name == "Tuned Classical RBF":

        gamma = config["gamma"]

        if gamma == "scale":
            gamma = 1.0 / (
                Xtr.shape[1] * Xtr.var()
            )

        Ktr = rbf_kernel(
            Xtr,
            gamma=gamma
        )

        Kval = rbf_kernel(
            Xval, Xtr,
            gamma=gamma
        )

        return Ktr, Kval

    raise ValueError("Unknown model")


Training-Only Model Selection

In [70]:

def p511_select_config(
    X_train_raw,
    y_train,
    model_name,
    seed
):
    inner_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    configs = p511_configs(model_name)

    scores_by_config = [
        [] for _ in configs
    ]

    for tr, val in inner_cv.split(
        X_train_raw, y_train
    ):

        Xtr, Xval = p5_transform(
            X_train_raw[tr],
            X_train_raw[val]
        )

        ytr = y_train[tr]
        yval = y_train[val]

        # Cache shared kernel components
        if model_name in [
            "Quantum Mixture",
            "Quantum Endpoint"
        ]:
            components = p5_get_kernel_components(
                Xtr, Xval, "quantum"
            )

        elif model_name == "Classical Mixture":
            components = p5_get_kernel_components(
                Xtr, Xval, "classical"
            )

        else:
            components = None

        for i, config in enumerate(configs):

            if components is not None:
                Ktr, Kval = p5_mix_kernels(
                    *components,
                    config["alpha"]
                )
            else:
                Ktr, Kval = p511_kernel_pair(
                    Xtr, Xval,
                    model_name, config
                )

            model = SVC(
                kernel="precomputed",
                C=config["C"]
            )

            model.fit(Ktr, ytr)

            scores = model.decision_function(Kval)

            scores_by_config[i].append(
                roc_auc_score(yval, scores)
            )

    mean_scores = [
        float(np.mean(scores))
        for scores in scores_by_config
    ]

    best_index = int(np.argmax(mean_scores))

    return configs[best_index], mean_scores[best_index]


Outer-Fold Model Evaluation

In [71]:

def p511_evaluate(
    X_train_raw,
    y_train,
    X_val_raw,
    y_val,
    model_name,
    config
):
    Xtr, Xval = p5_transform(
        X_train_raw,
        X_val_raw
    )

    Ktr, Kval = p511_kernel_pair(
        Xtr, Xval,
        model_name, config
    )

    model = SVC(
        kernel="precomputed",
        C=config["C"]
    )

    model.fit(Ktr, y_train)

    pred = model.predict(Kval)
    scores = model.decision_function(Kval)

    tn, fp, fn, tp = confusion_matrix(
        y_val, pred,
        labels=[0, 1]
    ).ravel()

    return {
        "roc_auc": roc_auc_score(
            y_val, scores
        ),
        "recall": recall_score(
            y_val, pred,
            zero_division=0
        ),
        "precision": precision_score(
            y_val, pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_val, pred,
            zero_division=0
        ),
        "false_negatives": int(fn),
        "false_positives": int(fp)
    }


Final Four-Model Comparison

In [72]:

p511_records = []

for n_samples in P511_SIZES:
    for seed in P511_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        outer_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            outer_cv.split(X_subset, y_subset)
        ):

            Xtr = X_subset[tr]
            Xval = X_subset[val]

            ytr = y_subset[tr]
            yval = y_subset[val]

            for model_name in P511_MODELS:

                start = time.perf_counter()

                config, inner_auc = p511_select_config(
                    Xtr, ytr,
                    model_name,
                    seed + fold
                )

                metrics = p511_evaluate(
                    Xtr, ytr,
                    Xval, yval,
                    model_name, config
                )

                elapsed = (
                    time.perf_counter() - start
                )

                p511_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "model": model_name,
                    "config": json.dumps(config),
                    "inner_cv_auc": inner_auc,
                    "runtime_seconds": elapsed,
                    **metrics
                })

                print(
                    f"n={n_samples}, seed={seed}, "
                    f"fold={fold}, {model_name}: "
                    f"AUC={metrics['roc_auc']:.4f}"
                )

        pd.DataFrame(p511_records).to_csv(
            P511_DIR / "phase511_raw.csv",
            index=False
        )

p511_df = pd.DataFrame(p511_records)

print("Phase 5.11 completed.")


n=40, seed=42, fold=0, Quantum Mixture: AUC=1.0000
n=40, seed=42, fold=0, Quantum Endpoint: AUC=1.0000
n=40, seed=42, fold=0, Classical Mixture: AUC=1.0000
n=40, seed=42, fold=0, Tuned Classical RBF: AUC=0.9778
n=40, seed=42, fold=1, Quantum Mixture: AUC=0.9250
n=40, seed=42, fold=1, Quantum Endpoint: AUC=0.9250
n=40, seed=42, fold=1, Classical Mixture: AUC=0.9500
n=40, seed=42, fold=1, Tuned Classical RBF: AUC=0.9500
n=40, seed=42, fold=2, Quantum Mixture: AUC=1.0000
n=40, seed=42, fold=2, Quantum Endpoint: AUC=0.9750
n=40, seed=42, fold=2, Classical Mixture: AUC=1.0000
n=40, seed=42, fold=2, Tuned Classical RBF: AUC=1.0000
n=40, seed=123, fold=0, Quantum Mixture: AUC=1.0000
n=40, seed=123, fold=0, Quantum Endpoint: AUC=0.9556
n=40, seed=123, fold=0, Classical Mixture: AUC=1.0000
n=40, seed=123, fold=0, Tuned Classical RBF: AUC=0.9556
n=40, seed=123, fold=1, Quantum Mixture: AUC=0.9000
n=40, seed=123, fold=1, Quantum Endpoint: AUC=0.9000
n=40, seed=123, fold=1, Classical Mixture: AUC=

Model Performance Summary

In [73]:

p511_per_seed = (
    p511_df
    .groupby(["n_samples", "seed", "model"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        runtime=("runtime_seconds", "sum")
    )
    .reset_index()
)

p511_summary = (
    p511_per_seed
    .groupby(["n_samples", "model"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
        precision_mean=("precision", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(p511_summary.round(4))

p511_summary.to_csv(
    P511_DIR / "phase511_summary.csv",
    index=False
)


,n_samples,model,auc_mean,auc_std,recall_mean,f1_mean,precision_mean,runtime_mean
0,40,Classical Mixture,0.9512,0.0285,0.7778,0.8064,0.9189,0.3500
1,40,Quantum Endpoint,0.9525,0.0139,0.7778,0.8243,0.9361,2.7918
2,40,Quantum Mixture,0.9627,0.0148,0.8222,0.8487,0.9398,2.9581
3,40,Tuned Classical RBF,0.9549,0.0253,0.2000,0.1781,0.1728,0.9909
4,80,Classical Mixture,0.9866,0.0057,0.8222,0.8802,0.9677,0.3501
5,80,Quantum Endpoint,0.9880,0.0084,0.7778,0.8469,0.9526,5.5073
6,80,Quantum Mixture,0.9835,0.0099,0.8222,0.8665,0.9308,5.9887
7,80,Tuned Classical RBF,0.9846,0.0048,0.4778,0.5203,0.6319,1.0567


Paired Performance Differences

In [74]:

paired = p511_df.pivot(
    index=["n_samples", "seed", "fold"],
    columns="model",
    values=["roc_auc", "recall", "f1"]
)

paired.columns = [
    f"{metric}__{model}"
    for metric, model in paired.columns
]

paired = paired.reset_index()

comparisons = [
    ("Quantum Mixture", "Quantum Endpoint"),
    ("Quantum Mixture", "Tuned Classical RBF"),
    ("Quantum Mixture", "Classical Mixture")
]

difference_records = []

for n_samples in P511_SIZES:

    subset = paired[
        paired["n_samples"] == n_samples
    ]

    for model_a, model_b in comparisons:

        record = {
            "n_samples": n_samples,
            "comparison": f"{model_a} - {model_b}"
        }

        for metric in ["roc_auc", "recall", "f1"]:

            difference = (
                subset[f"{metric}__{model_a}"]
                - subset[f"{metric}__{model_b}"]
            )

            record[f"{metric}_diff_mean"] = (
                difference.mean()
            )

            record[f"{metric}_diff_std"] = (
                difference.std()
            )

        difference_records.append(record)

p511_differences = pd.DataFrame(
    difference_records
)

display(p511_differences.round(4))

p511_differences.to_csv(
    P511_DIR / "phase511_differences.csv",
    index=False
)


,n_samples,comparison,roc_auc_diff_mean,roc_auc_diff_std,recall_diff_mean,recall_diff_std,f1_diff_mean,f1_diff_std
0,40,Quantum Mixture - Quantum Endpoint,0.0102,0.0164,0.0444,0.0882,0.0245,0.0486
1,40,Quantum Mixture - Tuned Classical RBF,0.0077,0.0341,0.6222,0.3667,0.6706,0.3599
2,40,Quantum Mixture - Classical Mixture,0.0114,0.0374,0.0444,0.0882,0.0423,0.0551
3,80,Quantum Mixture - Quantum Endpoint,-0.0045,0.0117,0.0444,0.0527,0.0196,0.0411
4,80,Quantum Mixture - Tuned Classical RBF,-0.0011,0.0103,0.3444,0.4667,0.3463,0.4758
5,80,Quantum Mixture - Classical Mixture,-0.0031,0.0130,-0.0000,0.0866,-0.0137,0.0587


Hyperparameter Selection Stability

In [75]:

selection_counts = (
    p511_df
    .groupby(["n_samples", "model", "config"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["n_samples", "model", "count"],
        ascending=[True, True, False]
    )
)

display(selection_counts)

selection_counts.to_csv(
    P511_DIR / "phase511_selected_configs.csv",
    index=False
)


,n_samples,model,config,count
0,40,Classical Mixture,"{""alpha"": 0, ""C"": 1.0}",6
2,40,Classical Mixture,"{""alpha"": 1.0, ""C"": 1.0}",2
1,40,Classical Mixture,"{""alpha"": 0.5, ""C"": 1.0}",1
4,40,Quantum Endpoint,"{""alpha"": 1.0, ""C"": 1.0}",5
3,40,Quantum Endpoint,"{""alpha"": 0.0, ""C"": 1.0}",4
6,40,Quantum Mixture,"{""alpha"": 0.25, ""C"": 1.0}",3
5,40,Quantum Mixture,"{""alpha"": 0, ""C"": 1.0}",2
9,40,Quantum Mixture,"{""alpha"": 1.0, ""C"": 1.0}",2
7,40,Quantum Mixture,"{""alpha"": 0.5, ""C"": 1.0}",1
8,40,Quantum Mixture,"{""alpha"": 0.75, ""C"": 1.0}",1


Threshold-Calibrated Benchmark Setup

In [76]:

import numpy as np
import pandas as pd
import time
import json

from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, recall_score,
    precision_score, f1_score,
    roc_auc_score, accuracy_score
)

P512_SIZES = [40, 80]
P512_SEEDS = [42, 123, 2026]
P512_TARGET_RECALL = 0.95

P512_MODELS = [
    "Quantum Mixture",
    "Quantum Endpoint",
    "Classical Mixture",
    "Tuned Classical RBF"
]

P512_DIR = Path("phase_5_results")
P512_DIR.mkdir(exist_ok=True)

print("Phase 5.12 ready.")


Phase 5.12 ready.


Training-Only Threshold Selection

In [77]:

def p512_choose_threshold(scores, y, target=0.95):
    scores = np.asarray(scores, dtype=float)
    y = np.asarray(y)

    if not np.isfinite(scores).all():
        raise ValueError("Non-finite decision scores")

    candidates = np.unique(scores)

    valid = []

    for threshold in candidates:
        pred = (scores >= threshold).astype(int)

        recall = recall_score(
            y, pred, zero_division=0
        )

        if recall >= target:
            tn, fp, fn, tp = confusion_matrix(
                y, pred, labels=[0, 1]
            ).ravel()

            specificity = tn / (tn + fp)

            valid.append(
                (specificity, threshold, recall)
            )

    if not valid:
        raise ValueError(
            "No threshold meets target recall"
        )

    # Highest specificity, then highest threshold
    specificity, threshold, recall = max(
        valid,
        key=lambda x: (x[0], x[1])
    )

    return float(threshold), float(recall)


Nested Calibration Predictions

In [78]:

def p512_oof_scores(
    X_train_raw,
    y_train,
    model_name,
    seed
):
    calibration_cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    oof_scores = np.zeros(len(y_train))

    for fold, (tr, val) in enumerate(
        calibration_cv.split(
            X_train_raw, y_train
        )
    ):

        # Select hyperparameters using only
        # calibration-fold training samples
        config, _ = p511_select_config(
            X_train_raw[tr],
            y_train[tr],
            model_name,
            seed + fold
        )

        # Fit preprocessing using only fold training
        Xtr, Xval = p5_transform(
            X_train_raw[tr],
            X_train_raw[val]
        )

        Ktr, Kval = p511_kernel_pair(
            Xtr, Xval,
            model_name,
            config
        )

        model = SVC(
            kernel="precomputed",
            C=config["C"]
        )

        model.fit(Ktr, y_train[tr])

        oof_scores[val] = (
            model.decision_function(Kval)
        )

    return oof_scores


Final Outer-Fold Evaluation

In [79]:

from sklearn.svm import SVC

def p512_evaluate(
    X_train_raw,
    y_train,
    X_val_raw,
    y_val,
    model_name,
    seed
):
    start = time.perf_counter()

    # Step 1: Nested training-only calibration
    oof_scores = p512_oof_scores(
        X_train_raw,
        y_train,
        model_name,
        seed
    )

    threshold, oof_recall = p512_choose_threshold(
        oof_scores,
        y_train,
        target=P512_TARGET_RECALL
    )

    # Step 2: Select final hyperparameters
    config, inner_auc = p511_select_config(
        X_train_raw,
        y_train,
        model_name,
        seed
    )

    # Step 3: Fit final model on outer training data
    Xtr, Xval = p5_transform(
        X_train_raw,
        X_val_raw
    )

    Ktr, Kval = p511_kernel_pair(
        Xtr, Xval,
        model_name,
        config
    )

    model = SVC(
        kernel="precomputed",
        C=config["C"]
    )

    model.fit(Ktr, y_train)

    scores = model.decision_function(Kval)

    predictions = (
        scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "config": json.dumps(config),
        "threshold": threshold,
        "oof_recall": oof_recall,
        "inner_cv_auc": inner_auc,
        "accuracy": accuracy_score(
            y_val, predictions
        ),
        "precision": precision_score(
            y_val, predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions,
            zero_division=0
        ),
        "specificity": (
            tn / (tn + fp)
        ),
        "f1": f1_score(
            y_val, predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_val, scores
        ),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "runtime_seconds": (
            time.perf_counter() - start
        )
    }


Fair Threshold-Calibrated Comparison

In [80]:

p512_records = []

for n_samples in P512_SIZES:
    for seed in P512_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        outer_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            outer_cv.split(X_subset, y_subset)
        ):

            Xtr = X_subset[tr]
            Xval = X_subset[val]

            ytr = y_subset[tr]
            yval = y_subset[val]

            for model_name in P512_MODELS:

                print(
                    f"Running n={n_samples}, "
                    f"seed={seed}, fold={fold}, "
                    f"model={model_name}"
                )

                metrics = p512_evaluate(
                    Xtr, ytr,
                    Xval, yval,
                    model_name,
                    seed + fold
                )

                p512_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "model": model_name,
                    **metrics
                })

        # Save after each seed
        pd.DataFrame(p512_records).to_csv(
            P512_DIR / "phase512_raw.csv",
            index=False
        )

p512_df = pd.DataFrame(p512_records)

print("Phase 5.12 completed.")


Running n=40, seed=42, fold=0, model=Quantum Mixture
Running n=40, seed=42, fold=0, model=Quantum Endpoint
Running n=40, seed=42, fold=0, model=Classical Mixture
Running n=40, seed=42, fold=0, model=Tuned Classical RBF
Running n=40, seed=42, fold=1, model=Quantum Mixture
Running n=40, seed=42, fold=1, model=Quantum Endpoint
Running n=40, seed=42, fold=1, model=Classical Mixture
Running n=40, seed=42, fold=1, model=Tuned Classical RBF
Running n=40, seed=42, fold=2, model=Quantum Mixture
Running n=40, seed=42, fold=2, model=Quantum Endpoint
Running n=40, seed=42, fold=2, model=Classical Mixture
Running n=40, seed=42, fold=2, model=Tuned Classical RBF
Running n=40, seed=123, fold=0, model=Quantum Mixture
Running n=40, seed=123, fold=0, model=Quantum Endpoint
Running n=40, seed=123, fold=0, model=Classical Mixture
Running n=40, seed=123, fold=0, model=Tuned Classical RBF
Running n=40, seed=123, fold=1, model=Quantum Mixture
Running n=40, seed=123, fold=1, model=Quantum Endpoint
Running n=4

Threshold-Calibrated Performance Summary

In [81]:

p512_per_seed = (
    p512_df
    .groupby(["n_samples", "seed", "model"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        specificity=("specificity", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        fp=("false_positives", "mean"),
        fn=("false_negatives", "mean"),
        runtime=("runtime_seconds", "sum")
    )
    .reset_index()
)

p512_summary = (
    p512_per_seed
    .groupby(["n_samples", "model"])
    .agg(
        auc_mean=("auc", "mean"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        specificity_mean=("specificity", "mean"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        fp_mean=("fp", "mean"),
        fn_mean=("fn", "mean"),
        runtime_mean=("runtime", "mean")
    )
    .reset_index()
)

display(p512_summary.round(4))

p512_summary.to_csv(
    P512_DIR / "phase512_summary.csv",
    index=False
)


,n_samples,model,auc_mean,recall_mean,recall_std,specificity_mean,precision_mean,f1_mean,fp_mean,fn_mean,runtime_mean
0,40,Classical Mixture,0.9512,0.9111,0.1018,0.7269,0.7626,0.7860,2.3333,0.4444,1.3449
1,40,Quantum Endpoint,0.9525,0.8444,0.0385,0.7917,0.7954,0.7677,1.7778,0.7778,6.7478
2,40,Quantum Mixture,0.9627,0.8667,0.0667,0.8333,0.8349,0.8077,1.4444,0.6667,8.6592
3,40,Tuned Classical RBF,0.9549,0.8889,0.1388,0.5293,0.6145,0.6842,3.8889,0.5556,3.5247
4,80,Classical Mixture,0.9866,0.9333,0.0333,0.9191,0.8945,0.9051,1.3333,0.6667,1.2380
5,80,Quantum Endpoint,0.9880,0.9000,0.0333,0.8999,0.8726,0.8743,1.6667,1.0000,9.8547
6,80,Quantum Mixture,0.9835,0.8889,0.0694,0.9130,0.8747,0.8708,1.4444,1.1111,10.1627
7,80,Tuned Classical RBF,0.9846,0.9778,0.0192,0.6585,0.6837,0.7886,5.6667,0.2222,2.8414


Quantum Mixture vs Classical Controls

In [82]:

paired = p512_df.pivot(
    index=["n_samples", "seed", "fold"],
    columns="model",
    values=["roc_auc", "recall", "specificity", "f1"]
)

paired.columns = [
    f"{metric}__{model}"
    for metric, model in paired.columns
]

paired = paired.reset_index()

controls = [
    "Quantum Endpoint",
    "Classical Mixture",
    "Tuned Classical RBF"
]

comparison_records = []

for n_samples in P512_SIZES:

    subset = paired[
        paired["n_samples"] == n_samples
    ]

    for control in controls:

        row = {
            "n_samples": n_samples,
            "comparison": (
                "Quantum Mixture - " + control
            )
        }

        for metric in [
            "roc_auc", "recall",
            "specificity", "f1"
        ]:

            difference = (
                subset[f"{metric}__Quantum Mixture"]
                - subset[f"{metric}__{control}"]
            )

            row[f"{metric}_difference"] = (
                difference.mean()
            )

        comparison_records.append(row)

p512_differences = pd.DataFrame(
    comparison_records
)

display(p512_differences.round(4))

p512_differences.to_csv(
    P512_DIR / "phase512_differences.csv",
    index=False
)


,n_samples,comparison,roc_auc_difference,recall_difference,specificity_difference,f1_difference
0,40,Quantum Mixture - Quantum Endpoint,0.0102,0.0222,0.0417,0.0401
1,40,Quantum Mixture - Classical Mixture,0.0114,-0.0444,0.1065,0.0217
2,40,Quantum Mixture - Tuned Classical RBF,0.0077,-0.0222,0.3040,0.1235
3,80,Quantum Mixture - Quantum Endpoint,-0.0045,-0.0111,0.0131,-0.0035
4,80,Quantum Mixture - Classical Mixture,-0.0031,-0.0444,-0.0061,-0.0343
5,80,Quantum Mixture - Tuned Classical RBF,-0.0011,-0.0889,0.2545,0.0823


Matched Operating-Point Setup

In [83]:

import numpy as np
import pandas as pd
import time
import json

from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
)

P513_SIZES = [40, 80]
P513_SEEDS = [42, 123, 2026]
P513_TARGETS = [0.90, 0.95]

P513_MODELS = [
    "Quantum Mixture",
    "Quantum Endpoint",
    "Classical Mixture",
    "Tuned Classical RBF"
]

P513_DIR = Path("phase_5_results")
P513_DIR.mkdir(exist_ok=True)

print("Phase 5.13 ready.")


Phase 5.13 ready.


Select Thresholds Without Validation Leakage

In [84]:

def p513_thresholds(oof_scores, y_train):

    thresholds = {}

    for target in P513_TARGETS:

        threshold, achieved_recall = (
            p512_choose_threshold(
                oof_scores,
                y_train,
                target=target
            )
        )

        thresholds[target] = {
            "threshold": threshold,
            "oof_recall": achieved_recall
        }

    return thresholds


Classical vs Quantum Operating Points

In [85]:

p513_records = []

for n_samples in P513_SIZES:
    for seed in P513_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        outer_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            outer_cv.split(X_subset, y_subset)
        ):

            Xtr_raw = X_subset[tr]
            Xval_raw = X_subset[val]

            ytr = y_subset[tr]
            yval = y_subset[val]

            for model_name in P513_MODELS:

                start = time.perf_counter()

                # Training-only calibration scores
                oof_scores = p512_oof_scores(
                    Xtr_raw,
                    ytr,
                    model_name,
                    seed + fold
                )

                thresholds = p513_thresholds(
                    oof_scores,
                    ytr
                )

                # Select final hyperparameters
                config, inner_auc = p511_select_config(
                    Xtr_raw,
                    ytr,
                    model_name,
                    seed + fold
                )

                # Fit preprocessing on outer training only
                Xtr, Xval = p5_transform(
                    Xtr_raw,
                    Xval_raw
                )

                Ktr, Kval = p511_kernel_pair(
                    Xtr, Xval,
                    model_name,
                    config
                )

                model = SVC(
                    kernel="precomputed",
                    C=config["C"]
                )

                model.fit(Ktr, ytr)

                scores = model.decision_function(Kval)

                for target, threshold_info in (
                    thresholds.items()
                ):

                    threshold = threshold_info[
                        "threshold"
                    ]

                    pred = (
                        scores >= threshold
                    ).astype(int)

                    tn, fp, fn, tp = confusion_matrix(
                        yval, pred,
                        labels=[0, 1]
                    ).ravel()

                    p513_records.append({
                        "n_samples": n_samples,
                        "seed": seed,
                        "fold": fold,
                        "model": model_name,
                        "target_recall": target,
                        "config": json.dumps(config),
                        "threshold": threshold,
                        "oof_recall": threshold_info[
                            "oof_recall"
                        ],
                        "recall": recall_score(
                            yval, pred,
                            zero_division=0
                        ),
                        "specificity": tn / (tn + fp),
                        "precision": precision_score(
                            yval, pred,
                            zero_division=0
                        ),
                        "f1": f1_score(
                            yval, pred,
                            zero_division=0
                        ),
                        "roc_auc": roc_auc_score(
                            yval, scores
                        ),
                        "false_negatives": int(fn),
                        "false_positives": int(fp),
                        "runtime_seconds": (
                            time.perf_counter() - start
                        )
                    })

                print(
                    f"Completed n={n_samples}, "
                    f"seed={seed}, fold={fold}, "
                    f"{model_name}"
                )

        # Save after every seed
        pd.DataFrame(p513_records).to_csv(
            P513_DIR / "phase513_raw.csv",
            index=False
        )

p513_df = pd.DataFrame(p513_records)

print("Phase 5.13 completed.")


Completed n=40, seed=42, fold=0, Quantum Mixture
Completed n=40, seed=42, fold=0, Quantum Endpoint
Completed n=40, seed=42, fold=0, Classical Mixture
Completed n=40, seed=42, fold=0, Tuned Classical RBF
Completed n=40, seed=42, fold=1, Quantum Mixture
Completed n=40, seed=42, fold=1, Quantum Endpoint
Completed n=40, seed=42, fold=1, Classical Mixture
Completed n=40, seed=42, fold=1, Tuned Classical RBF
Completed n=40, seed=42, fold=2, Quantum Mixture
Completed n=40, seed=42, fold=2, Quantum Endpoint
Completed n=40, seed=42, fold=2, Classical Mixture
Completed n=40, seed=42, fold=2, Tuned Classical RBF
Completed n=40, seed=123, fold=0, Quantum Mixture
Completed n=40, seed=123, fold=0, Quantum Endpoint
Completed n=40, seed=123, fold=0, Classical Mixture
Completed n=40, seed=123, fold=0, Tuned Classical RBF
Completed n=40, seed=123, fold=1, Quantum Mixture
Completed n=40, seed=123, fold=1, Quantum Endpoint
Completed n=40, seed=123, fold=1, Classical Mixture
Completed n=40, seed=123, fold=

Operating-Point Performance Summary

In [86]:

# Average folds within each seed
p513_per_seed = (
    p513_df
    .groupby([
        "n_samples",
        "seed",
        "model",
        "target_recall"
    ])
    .agg(
        recall=("recall", "mean"),
        specificity=("specificity", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean"),
        auc=("roc_auc", "mean")
    )
    .reset_index()
)

# Then aggregate across seeds
p513_summary = (
    p513_per_seed
    .groupby([
        "n_samples",
        "model",
        "target_recall"
    ])
    .agg(
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        specificity_mean=("specificity", "mean"),
        specificity_std=("specificity", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        auc_mean=("auc", "mean")
    )
    .reset_index()
)

display(p513_summary.round(4))

p513_summary.to_csv(
    P513_DIR / "phase513_summary.csv",
    index=False
)


,n_samples,model,target_recall,recall_mean,recall_std,specificity_mean,specificity_std,precision_mean,f1_mean,auc_mean
0,40,Classical Mixture,0.90,0.8000,0.0667,0.8796,0.0350,0.8433,0.7796,0.9512
1,40,Classical Mixture,0.95,0.9111,0.1018,0.7269,0.0893,0.7626,0.7860,0.9512
2,40,Quantum Endpoint,0.90,0.8222,0.0385,0.8935,0.0401,0.8710,0.8123,0.9525
3,40,Quantum Endpoint,0.95,0.8444,0.0385,0.7917,0.0362,0.7954,0.7677,0.9525
4,40,Quantum Mixture,0.90,0.8444,0.0770,0.9460,0.0255,0.9259,0.8618,0.9627
5,40,Quantum Mixture,0.95,0.8667,0.0667,0.8333,0.0362,0.8349,0.8077,0.9627
6,40,Tuned Classical RBF,0.90,0.6889,0.2341,0.6404,0.1898,0.4884,0.5286,0.9549
7,40,Tuned Classical RBF,0.95,0.8889,0.1388,0.5293,0.1599,0.6145,0.6842,0.9549
8,80,Classical Mixture,0.90,0.9000,0.0333,0.9261,0.0410,0.8999,0.8883,0.9866
9,80,Classical Mixture,0.95,0.9333,0.0333,0.9191,0.0340,0.8945,0.9051,0.9866


Paired Operating-Point Differences

In [87]:

paired = p513_df.pivot(
    index=[
        "n_samples",
        "seed",
        "fold",
        "target_recall"
    ],
    columns="model",
    values=[
        "recall",
        "specificity",
        "f1"
    ]
)

paired.columns = [
    f"{metric}__{model}"
    for metric, model in paired.columns
]

paired = paired.reset_index()

controls = [
    "Quantum Endpoint",
    "Classical Mixture",
    "Tuned Classical RBF"
]

records = []

for (n_samples, target), subset in paired.groupby(
    ["n_samples", "target_recall"]
):

    for control in controls:

        record = {
            "n_samples": n_samples,
            "target_recall": target,
            "comparison": (
                "Quantum Mixture - " + control
            )
        }

        for metric in [
            "recall",
            "specificity",
            "f1"
        ]:

            diff = (
                subset[
                    f"{metric}__Quantum Mixture"
                ]
                - subset[f"{metric}__{control}"]
            )

            record[f"{metric}_difference"] = (
                diff.mean()
            )

        records.append(record)

p513_differences = pd.DataFrame(records)

display(p513_differences.round(4))

p513_differences.to_csv(
    P513_DIR / "phase513_differences.csv",
    index=False
)


,n_samples,target_recall,comparison,recall_difference,specificity_difference,f1_difference
0,40,0.90,Quantum Mixture - Quantum Endpoint,0.0222,0.0525,0.0495
1,40,0.90,Quantum Mixture - Classical Mixture,0.0444,0.0664,0.0821
2,40,0.90,Quantum Mixture - Tuned Classical RBF,0.1556,0.3056,0.3332
3,40,0.95,Quantum Mixture - Quantum Endpoint,0.0222,0.0417,0.0401
4,40,0.95,Quantum Mixture - Classical Mixture,-0.0444,0.1065,0.0217
5,40,0.95,Quantum Mixture - Tuned Classical RBF,-0.0222,0.3040,0.1235
6,80,0.90,Quantum Mixture - Quantum Endpoint,0.0333,0.0000,0.0216
7,80,0.90,Quantum Mixture - Classical Mixture,-0.0111,0.0131,0.0021
8,80,0.90,Quantum Mixture - Tuned Classical RBF,-0.0889,0.2741,0.0996
9,80,0.95,Quantum Mixture - Quantum Endpoint,-0.0111,0.0131,-0.0035


Calibration Reliability

In [88]:

calibration_check = (
    p513_df
    .assign(
        target_met=lambda d:
            d["recall"] >= d["target_recall"]
    )
    .groupby([
        "n_samples",
        "model",
        "target_recall"
    ])
    .agg(
        target_met_rate=("target_met", "mean"),
        mean_oof_recall=("oof_recall", "mean"),
        mean_validation_recall=("recall", "mean")
    )
    .reset_index()
)

display(calibration_check.round(4))

calibration_check.to_csv(
    P513_DIR / "phase513_calibration_reliability.csv",
    index=False
)


,n_samples,model,target_recall,target_met_rate,mean_oof_recall,mean_validation_recall
0,40,Classical Mixture,0.90,0.4444,0.90,0.8000
1,40,Classical Mixture,0.95,0.7778,1.00,0.9111
2,40,Quantum Endpoint,0.90,0.5556,0.90,0.8222
3,40,Quantum Endpoint,0.95,0.6667,1.00,0.8444
4,40,Quantum Mixture,0.90,0.5556,0.90,0.8444
5,40,Quantum Mixture,0.95,0.6667,1.00,0.8667
6,40,Tuned Classical RBF,0.90,0.5556,0.90,0.6889
7,40,Tuned Classical RBF,0.95,0.6667,1.00,0.8889
8,80,Classical Mixture,0.90,0.7778,0.90,0.9000
9,80,Classical Mixture,0.95,0.5556,0.95,0.9333


Polynomial SVM Setup

In [89]:

import numpy as np
import pandas as pd
import json
import time

from pathlib import Path
from itertools import product

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

P514_SIZES = [40, 80]
P514_SEEDS = [42, 123, 2026]
P514_TARGETS = [0.90, 0.95]

P514_C = [0.1, 1, 10]
P514_DEGREES = [2, 3, 4]
P514_GAMMAS = ["scale", 0.1, 1.0]

P514_DIR = Path("phase_5_results")
P514_DIR.mkdir(exist_ok=True)

print("Phase 5.14 ready.")


Phase 5.14 ready.


Training-Only Polynomial Model Selection

In [90]:

def p514_select_config(X, y, seed):

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    configs = [
        {"C": C, "degree": degree, "gamma": gamma}
        for C, degree, gamma in product(
            P514_C,
            P514_DEGREES,
            P514_GAMMAS
        )
    ]

    best_auc = -np.inf
    best_config = None

    for config in configs:

        aucs = []

        for tr, val in cv.split(X, y):

            Xtr, Xval = p5_transform(
                X[tr], X[val]
            )

            model = SVC(
                kernel="poly",
                C=config["C"],
                degree=config["degree"],
                gamma=config["gamma"],
                coef0=1.0
            )

            model.fit(Xtr, y[tr])

            scores = model.decision_function(Xval)

            aucs.append(
                roc_auc_score(y[val], scores)
            )

        mean_auc = float(np.mean(aucs))

        if mean_auc > best_auc:
            best_auc = mean_auc
            best_config = config.copy()

    return best_config, best_auc


Leakage-Safe Polynomial Threshold Calibration

In [91]:

def p514_oof_scores(X, y, seed):

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=seed
    )

    oof = np.zeros(len(y))

    for fold, (tr, val) in enumerate(
        cv.split(X, y)
    ):

        config, _ = p514_select_config(
            X[tr],
            y[tr],
            seed + fold
        )

        Xtr, Xval = p5_transform(
            X[tr], X[val]
        )

        model = SVC(
            kernel="poly",
            C=config["C"],
            degree=config["degree"],
            gamma=config["gamma"],
            coef0=1.0
        )

        model.fit(Xtr, y[tr])

        oof[val] = model.decision_function(Xval)

    return oof


Polynomial SVM Outer-Fold Evaluation

In [92]:

p514_records = []

for n_samples in P514_SIZES:
    for seed in P514_SEEDS:

        path = Path(
            f"limited_data/data_n{n_samples}_seed{seed}.npz"
        )

        with np.load(path) as data:
            subset_idx = data["train_idx"].copy()

        X_subset = X_raw[subset_idx]
        y_subset = y_all[subset_idx]

        outer_cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed
        )

        for fold, (tr, val) in enumerate(
            outer_cv.split(X_subset, y_subset)
        ):

            Xtr_raw = X_subset[tr]
            Xval_raw = X_subset[val]

            ytr = y_subset[tr]
            yval = y_subset[val]

            start = time.perf_counter()

            # Training-only hyperparameter selection
            config, inner_auc = p514_select_config(
                Xtr_raw,
                ytr,
                seed + fold
            )

            # Nested calibration
            oof_scores = p514_oof_scores(
                Xtr_raw,
                ytr,
                seed + fold
            )

            thresholds = {
                "default": 0.0
            }

            for target in P514_TARGETS:

                threshold, _ = p512_choose_threshold(
                    oof_scores,
                    ytr,
                    target=target
                )

                thresholds[target] = threshold

            # Fit final model on outer training only
            Xtr, Xval = p5_transform(
                Xtr_raw,
                Xval_raw
            )

            model = SVC(
                kernel="poly",
                C=config["C"],
                degree=config["degree"],
                gamma=config["gamma"],
                coef0=1.0
            )

            model.fit(Xtr, ytr)

            scores = model.decision_function(Xval)
            auc = roc_auc_score(yval, scores)

            elapsed = time.perf_counter() - start

            for target, threshold in thresholds.items():

                pred = (
                    scores >= threshold
                ).astype(int)

                tn, fp, fn, tp = confusion_matrix(
                    yval, pred,
                    labels=[0, 1]
                ).ravel()

                p514_records.append({
                    "n_samples": n_samples,
                    "seed": seed,
                    "fold": fold,
                    "model": "Tuned Polynomial SVM",
                    "target": str(target),
                    "config": json.dumps(config),
                    "threshold": threshold,
                    "roc_auc": auc,
                    "recall": recall_score(
                        yval, pred,
                        zero_division=0
                    ),
                    "specificity": tn / (tn + fp),
                    "precision": precision_score(
                        yval, pred,
                        zero_division=0
                    ),
                    "f1": f1_score(
                        yval, pred,
                        zero_division=0
                    ),
                    "false_negatives": int(fn),
                    "false_positives": int(fp),
                    "runtime_seconds": elapsed
                })

            print(
                f"Completed n={n_samples}, "
                f"seed={seed}, fold={fold}"
            )

        pd.DataFrame(p514_records).to_csv(
            P514_DIR / "phase514_raw.csv",
            index=False
        )

p514_df = pd.DataFrame(p514_records)

print("Polynomial benchmark completed.")


Completed n=40, seed=42, fold=0
Completed n=40, seed=42, fold=1
Completed n=40, seed=42, fold=2
Completed n=40, seed=123, fold=0
Completed n=40, seed=123, fold=1
Completed n=40, seed=123, fold=2
Completed n=40, seed=2026, fold=0
Completed n=40, seed=2026, fold=1
Completed n=40, seed=2026, fold=2
Completed n=80, seed=42, fold=0
Completed n=80, seed=42, fold=1
Completed n=80, seed=42, fold=2
Completed n=80, seed=123, fold=0
Completed n=80, seed=123, fold=1
Completed n=80, seed=123, fold=2
Completed n=80, seed=2026, fold=0
Completed n=80, seed=2026, fold=1
Completed n=80, seed=2026, fold=2
Polynomial benchmark completed.


Polynomial SVM Results

In [93]:

p514_per_seed = (
    p514_df
    .groupby(["n_samples", "seed", "target"])
    .agg(
        auc=("roc_auc", "mean"),
        recall=("recall", "mean"),
        specificity=("specificity", "mean"),
        precision=("precision", "mean"),
        f1=("f1", "mean")
    )
    .reset_index()
)

p514_summary = (
    p514_per_seed
    .groupby(["n_samples", "target"])
    .agg(
        auc_mean=("auc", "mean"),
        auc_std=("auc", "std"),
        recall_mean=("recall", "mean"),
        specificity_mean=("specificity", "mean"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean")
    )
    .reset_index()
)

display(p514_summary.round(4))

p514_summary.to_csv(
    P514_DIR / "phase514_summary.csv",
    index=False
)


,n_samples,target,auc_mean,auc_std,recall_mean,specificity_mean,precision_mean,f1_mean
0,40,0.9,0.9818,0.0107,0.6889,0.8210,0.6005,0.6185
1,40,0.95,0.9818,0.0107,0.9111,0.5139,0.6413,0.6941
2,40,default,0.9818,0.0107,0.5778,0.9738,0.8333,0.6491
3,80,0.9,0.9894,0.0068,0.8778,0.9383,0.9124,0.8865
4,80,0.95,0.9894,0.0068,0.8778,0.9383,0.9124,0.8865
5,80,default,0.9894,0.0068,0.7111,0.9796,0.9664,0.7774


Quantum Mixture vs Polynomial SVM

In [94]:

quantum_results = p513_df[
    p513_df["model"] == "Quantum Mixture"
].copy()

polynomial_results = p514_df[
    p514_df["target"] != "default"
].copy()

polynomial_results["target_recall"] = (
    polynomial_results["target"].astype(float)
)

paired = quantum_results.merge(
    polynomial_results,
    on=[
        "n_samples",
        "seed",
        "fold",
        "target_recall"
    ],
    suffixes=("_quantum", "_polynomial"),
    validate="one_to_one"
)

for metric in [
    "roc_auc",
    "recall",
    "specificity",
    "f1"
]:
    paired[f"{metric}_difference"] = (
        paired[f"{metric}_quantum"]
        - paired[f"{metric}_polynomial"]
    )

p514_comparison = (
    paired
    .groupby(["n_samples", "target_recall"])
    [[
        "roc_auc_difference",
        "recall_difference",
        "specificity_difference",
        "f1_difference"
    ]]
    .mean()
    .reset_index()
)

display(p514_comparison.round(4))

p514_comparison.to_csv(
    P514_DIR / "phase514_quantum_comparison.csv",
    index=False
)


,n_samples,target_recall,roc_auc_difference,recall_difference,specificity_difference,f1_difference
0,40,0.90,-0.0191,0.1556,0.1250,0.2432
1,40,0.95,-0.0191,-0.0444,0.3194,0.1136
2,80,0.90,-0.0059,0.0111,0.0008,0.0039
3,80,0.95,-0.0059,0.0111,-0.0253,-0.0156
